# Medallion Extraction Pipeline

Colab-first paper ETL: **Bronze** captures the PDF with Docling, **Silver** cleans it, gates out non-data assets and normalises what survives, **Gold** assembles validated records, and a local SQLite mirrors the production schema. Gold's one model call goes to a local Ollama server.

In [1]:
import gc
import json
import math
import os
import re
import shutil
import subprocess
import sys
import tempfile
import unicodedata
import urllib.error
import urllib.request
from dataclasses import dataclass, field
from functools import cached_property, lru_cache
from itertools import chain, zip_longest
from pathlib import Path
from typing import Any, NamedTuple, Optional, Sequence

def run(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

# Must precede the first torch import: the allocator reads it once, at CUDA init.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
    CUDA_VERSION = torch.version.cuda
except ImportError:
    torch = None
    CUDA_AVAILABLE = False
    CUDA_VERSION = None

print(f"CUDA Available : {CUDA_AVAILABLE}")
print(f"CUDA Version   : {CUDA_VERSION}")

run([sys.executable, "-m", "pip", "install", "-q",
    "docling", "pymupdf", "polars", "pandas", "numpy", "pillow",
    "pydantic>=2", "sqlalchemy"])

# PP-Chart2Table turns a chart image back into a table; without it a figure cannot be
# tested against the schema at all (see REQUIRE_FIGURE_DATA).
if CUDA_AVAILABLE:
    try:
        run([sys.executable, "-m", "pip", "install", "-q", "paddleocr", "paddlepaddle-gpu"])
    except subprocess.CalledProcessError:
        run([sys.executable, "-m", "pip", "install", "-q", "paddleocr", "paddlepaddle"])
else:
    run([sys.executable, "-m", "pip", "install", "-q", "paddleocr", "paddlepaddle"])

import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

WORKDIR = Path("/content") if Path("/content").exists() else Path.cwd()
PROJECT_ROOT = WORKDIR / "medallion_extraction"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
BRONZE_ROOT = ARTIFACT_ROOT / "bronze"
SILVER_ROOT = ARTIFACT_ROOT / "silver"
for _root in (PROJECT_ROOT, BRONZE_ROOT, SILVER_ROOT):
    _root.mkdir(parents=True, exist_ok=True)

LOCAL_DB_PATH = PROJECT_ROOT / "extraction_local.sqlite"

# ── Gold: a local Ollama server, and nothing else ─────────────────────────────
# No hosted provider and no API key: the paper never leaves this machine.
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gpt-oss:20b")
# Sent per request: Ollama's default is small and slides the window silently rather than
# erroring, dropping the head of the prompt — the methods prose Gold reads.
OLLAMA_NUM_CTX = int(os.getenv("OLLAMA_NUM_CTX", "32768"))
# gpt-oss reasons before answering; Ollama returns that in `message.thinking` and only
# `message.content` is parsed. "low" is enough: the prompt asks for two fields.
OLLAMA_THINK = os.getenv("OLLAMA_THINK", "low")
OLLAMA_TIMEOUT = int(os.getenv("OLLAMA_TIMEOUT", "600"))

NATIVE_TEXT_THRESHOLD = int(os.getenv("NATIVE_TEXT_THRESHOLD", "150"))

# ── Bronze cost/fidelity knobs ────────────────────────────────────────────────
TABLE_STRUCTURE = os.getenv("TABLE_STRUCTURE", "1") == "1"
TABLE_STRUCTURE_MODE = os.getenv("TABLE_STRUCTURE_MODE", "fast")   # fast | accurate
IMAGES_SCALE = float(os.getenv("IMAGES_SCALE", "2.0"))             # <2.0 blurs axis ticks

# ── Silver gate knobs ─────────────────────────────────────────────────────────
# 1 keeps figures no chart converter could test out of Gold; 0 admits them on the probe.
REQUIRE_FIGURE_DATA = os.getenv("REQUIRE_FIGURE_DATA", "1") == "1"
# Figures per chart-model call. The attention tensor scales with the pixel count and
# Docling's models are still resident, so a whole paper in one call OOMs on a 16 GB card;
# `convert_figures` escalates from here if it still does.
CHART_BATCH_SIZE = int(os.getenv("CHART_BATCH_SIZE", "2"))
# Long side of the copy the chart model is shown. PP-Chart2Table tiles its input, so cost
# follows pixels; the archived PNG keeps its IMAGES_SCALE fidelity either way. 1024 is the
# model's own native input size, and CHART_MIN_PIXELS is where tick labels stop being
# legible, so `convert_figures` will not shrink past it.
CHART_MAX_PIXELS = int(os.getenv("CHART_MAX_PIXELS", "1024"))
CHART_MIN_PIXELS = int(os.getenv("CHART_MIN_PIXELS", "512"))
# Last resort: convert on the CPU, minutes per figure, rather than lose the paper.
CHART_CPU_FALLBACK = os.getenv("CHART_CPU_FALLBACK", "1") == "1"
FIGURE_CONTEXT_CHARS = int(os.getenv("FIGURE_CONTEXT_CHARS", "1200"))
MAX_PROMPT_OBSERVATIONS = int(os.getenv("MAX_PROMPT_OBSERVATIONS", "120"))

print(f"Project root : {PROJECT_ROOT}")
print(f"SQLite DB    : {LOCAL_DB_PATH}")
print(f"Ollama       : {OLLAMA_MODEL} at {OLLAMA_HOST} (num_ctx {OLLAMA_NUM_CTX})")
print(f"Table struct : {TABLE_STRUCTURE} ({TABLE_STRUCTURE_MODE})")

CUDA Available : True
CUDA Version   : 12.8
$ /home/zeus/miniconda3/envs/cloudspace/bin/python -m pip install -q docling pymupdf polars pandas numpy pillow pydantic>=2 sqlalchemy
$ /home/zeus/miniconda3/envs/cloudspace/bin/python -m pip install -q paddleocr paddlepaddle-gpu
Project root : /content/medallion_extraction
SQLite DB    : /content/medallion_extraction/extraction_local.sqlite
Ollama       : gpt-oss:20b at http://localhost:11434 (num_ctx 32768)
Table struct : True (fast)


## Bronze

Raw Docling capture: markdown, one CSV per table and one PNG per figure, plus a manifest every later stage reads.

In [2]:
import hashlib
from datetime import datetime, timezone

import fitz
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling_core.types.doc import PictureItem, TableItem

try:
    from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
except ImportError as exc:  # docling < 2.26 kept these in pipeline_options
    raise ImportError(
        "docling >= 2.26 is required: AcceleratorOptions moved to "
        "docling.datamodel.accelerator_options. Importing the old location instead "
        "silently changes which pipeline options are honoured, so this stops here. "
        "Run: pip install -U 'docling>=2.26'"
    ) from exc


def file_hash(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def pdf_looks_native(pdf_path: Path, sample_pages: int = 2) -> bool:
    # `with` matters: an un-closed handle leaks a descriptor per paper and keeps the
    # PDF locked on Windows for the rest of the session.
    try:
        with fitz.open(pdf_path) as document:
            sample = [document[index].get_text("text") or ""
                      for index in range(min(sample_pages, len(document)))]
    except (fitz.FileDataError, fitz.EmptyFileError, RuntimeError, OSError):
        return False   # an unreadable PDF is not a native one; Docling will OCR it
    return sum(len(page.strip()) for page in sample) >= NATIVE_TEXT_THRESHOLD


# A DocumentConverter loads the layout and table models on construction, so it is
# cached by configuration: once per kernel rather than once per paper.
_CONVERTER_CACHE: dict = {}


def build_docling_converter(*, native_pdf: bool, table_structure: bool = TABLE_STRUCTURE):
    key = (native_pdf, table_structure, TABLE_STRUCTURE_MODE, IMAGES_SCALE)
    flags = {
        "native_pdf": native_pdf,
        "ocr_enabled": not native_pdf,
        "table_structure_enabled": table_structure,
        "table_structure_mode": TABLE_STRUCTURE_MODE if table_structure else None,
        "images_scale": IMAGES_SCALE,
        "converter_cached": key in _CONVERTER_CACHE,
    }
    if key in _CONVERTER_CACHE:
        return _CONVERTER_CACHE[key], flags

    options = PdfPipelineOptions()
    options.do_ocr = not native_pdf
    options.do_table_structure = table_structure
    if table_structure:
        options.table_structure_options.mode = (
            TableFormerMode.ACCURATE if TABLE_STRUCTURE_MODE == "accurate" else TableFormerMode.FAST
        )
        options.table_structure_options.do_cell_matching = True
    options.generate_picture_images = True
    options.generate_page_images = False
    options.images_scale = IMAGES_SCALE
    options.accelerator_options = AcceleratorOptions(
        device=AcceleratorDevice.CUDA if CUDA_AVAILABLE else AcceleratorDevice.CPU,
        num_threads=8,
    )
    converter = DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=options)}
    )
    _CONVERTER_CACHE[key] = converter
    return converter, flags


def caption_text(element: Any, document: Any) -> Optional[str]:
    """Resolve a caption to text. `element.captions` holds references, not strings."""
    getter = getattr(element, "caption_text", None)
    if callable(getter):
        try:
            text = getter(document)
        except (TypeError, AttributeError, KeyError):
            text = None       # older docling takes no argument, or has no captions
        if text and str(text).strip():
            return str(text).strip()

    parts = []
    for reference in getattr(element, "captions", None) or []:
        text = getattr(reference, "text", None)
        if text is None:
            try:
                text = getattr(reference.resolve(document), "text", None)
            except (AttributeError, KeyError, ValueError):
                text = None   # a dangling reference into the document tree
        if text:
            parts.append(str(text))
    return " ".join(parts).strip() or None


def _page_number(element: Any) -> Optional[int]:
    provenance = getattr(element, "prov", None) or []
    if not provenance:
        return None
    try:
        return int(provenance[0].page_no)
    except (AttributeError, TypeError, ValueError):
        return None


def extract_bronze(pdf_path: Path) -> dict:
    slug = pdf_path.stem
    paper_root = BRONZE_ROOT / slug
    if paper_root.exists():
        shutil.rmtree(paper_root)
    tables_root = paper_root / "tables"
    figures_root = paper_root / "figures"
    tables_root.mkdir(parents=True, exist_ok=True)
    figures_root.mkdir(parents=True, exist_ok=True)

    native_pdf = pdf_looks_native(pdf_path)
    converter, docling_flags = build_docling_converter(native_pdf=native_pdf)

    started = datetime.now(timezone.utc)
    print(f"Bronze: {slug} | native_pdf={native_pdf} | ocr={docling_flags['ocr_enabled']} "
          f"| tables={docling_flags['table_structure_enabled']} "
          f"| converter_cached={docling_flags['converter_cached']}")
    document = converter.convert(pdf_path).document

    markdown_path = paper_root / f"{slug}.md"
    markdown_path.write_text(document.export_to_markdown(), encoding="utf-8")

    table_records = []
    figure_records = []
    text_records = []
    table_index = 0
    figure_index = 0
    current_heading = None

    # `order` is the reading position: Silver finds the text nearest a figure with it.
    for order, (element, _level) in enumerate(document.iterate_items()):
        item_ref = str(getattr(element, "self_ref", None) or f"#/unknown/{order}")
        page_number = _page_number(element)

        # Dispatch on type, not on attributes: several Docling classes expose
        # `get_image`, so duck typing pulls in page furniture alongside real figures.
        if isinstance(element, TableItem):
            table_index += 1
            record = {
                "table_index": table_index,
                "order": order,
                "docling_item_ref": item_ref,
                "page_number": page_number,
                "caption": caption_text(element, document),
            }
            try:
                try:
                    frame = element.export_to_dataframe(document)
                except TypeError:
                    frame = element.export_to_dataframe()
                table_path = tables_root / f"table_{table_index:03d}.csv"
                frame.to_csv(table_path, index=False)
                record.update({
                    "csv_path": str(table_path),
                    "row_count": int(frame.shape[0]),
                    "col_count": int(frame.shape[1]),
                })
            except (ValueError, TypeError, KeyError, OSError) as exc:
                record.update({"csv_path": None, "error": f"{type(exc).__name__}: {exc}"})
            table_records.append(record)

        elif isinstance(element, PictureItem):
            figure_index += 1
            record = {
                "figure_index": figure_index,
                "order": order,
                "docling_item_ref": item_ref,
                "page_number": page_number,
                "caption": caption_text(element, document),
            }
            try:
                image = element.get_image(document)
                if image is None:
                    record.update({"image_path": None, "error": "no image payload"})
                else:
                    image_path = figures_root / f"figure_{figure_index:03d}.png"
                    image.save(image_path)
                    record.update({
                        "image_path": str(image_path),
                        "width": int(image.width),
                        "height": int(image.height),
                    })
            except (ValueError, TypeError, KeyError, OSError) as exc:
                record.update({"image_path": None, "error": f"{type(exc).__name__}: {exc}"})
            figure_records.append(record)

        else:
            raw_text = str(getattr(element, "text", "") or "").strip()
            label = str(getattr(element, "label", "") or "").lower()
            if "heading" in label or "section_header" in label:
                current_heading = raw_text or current_heading
            if raw_text:
                text_records.append({
                    "order": order,
                    "docling_item_ref": item_ref,
                    "page_number": page_number,
                    "heading": current_heading,
                    "text": raw_text,
                })

    finished = datetime.now(timezone.utc)
    manifest = {
        "paper_slug": slug,
        "source_pdf": str(pdf_path),
        "file_hash": file_hash(pdf_path),
        "native_pdf": native_pdf,
        "docling_flags": docling_flags,
        "page_count": len(getattr(document, "pages", {}) or {}),
        "markdown_path": str(markdown_path),
        "tables": table_records,
        "figures": figure_records,
        "texts": text_records,
        "elapsed_seconds": round((finished - started).total_seconds(), 2),
        "created_at": finished.isoformat(),
    }
    (paper_root / "manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
    )
    # Docling's models stay resident; returning their unused blocks is what leaves room
    # for the chart model in Silver.
    if torch is not None and CUDA_AVAILABLE:
        torch.cuda.empty_cache()

    print(f"        {len(table_records)} tables, {len(figure_records)} figures, "
          f"{len(text_records)} text items in {manifest['elapsed_seconds']}s")
    return manifest

## The Schema Gate

One structural contract for tables and chart-converted figures alike: an asset passes only if it has an ordered axis, labels and numeric values. No keyword list, so it survives a change of corpus.

In [3]:
# The gate: a measurement is an INDICATOR VALUE observed at a point on an ORDINAL AXIS,
# optionally grouped into SERIES. Nothing below names a domain, so one contract covers
# native tables, chart-converted figures and a corpus about something else. Which label
# names the indicator and which names the treatment is settled later, in Interpretation.
# `plot_likeness` only keeps chart conversion off photographs and logos.

# ─── Contract knobs: properties of the schema, not of the subject matter ──────

MIN_AXIS_POINTS = 3        # fewer than 3 points is not a series
MIN_SERIES_POINTS = 3      # an indicator must be observed at >= 3 axis points
MIN_NUMERIC_RATIO = 0.6    # share of parseable cells for a column to count as numeric
MIN_ENUMERATION_ROWS = 6   # rows a row-number column must track before it is one
EPS = 1e-9


# ─── Cell parsing ─────────────────────────────────────────────────────────────
# Only notation is stripped — dispersion, comparators, significance letters, decimal
# conventions — never meaning.

_SD_SPLIT = re.compile(r"\s*(?:±|\+/-|\+-)\s*")
_PAREN_TAIL = re.compile(r"\s*\([^)]*\)\s*$")
_LEAD_CMP = re.compile(r"^[<>≤≥~≈=]+\s*")
_TRAIL_MARK = re.compile(r"\s*(?:[*†‡§¶]+|[a-zA-Z]{1,3})$")
_THOUSANDS = re.compile(r"^[-+]?\d{1,3}(?:,\d{3})+$")
_DECIMAL_COMMA = re.compile(r"^[-+]?\d+,\d+$")


@lru_cache(maxsize=16384)
def _parse_number_text(value: str) -> Optional[float]:
    text = value.replace("−", "-").replace(" ", " ").strip()
    if not text:
        return None

    text = _SD_SPLIT.split(text, 1)[0].strip()   # 6.35 ± 0.12  -> 6.35
    text = _PAREN_TAIL.sub("", text).strip()     # 6.35 (0.12)  -> 6.35
    text = _LEAD_CMP.sub("", text).strip()       # <0.1         -> 0.1
    text = text.rstrip("%").strip()
    text = _TRAIL_MARK.sub("", text).strip()     # 6.35b, 3 d   -> 6.35, 3
    if not text:
        return None

    if _THOUSANDS.match(text):
        text = text.replace(",", "")
    elif _DECIMAL_COMMA.match(text):
        text = text.replace(",", ".")

    try:
        number = float(text)
    except ValueError:
        return None
    return number if math.isfinite(number) else None


def parse_number(value: Any) -> Optional[float]:
    """Parse a table cell to a float, or None. The string path is cached: the gate, the
    reference test and the composition reader all re-parse the same cells."""
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        number = float(value)
        return number if math.isfinite(number) else None
    return _parse_number_text(str(value).strip())


# Looser than cell parsing: a wide axis lives in the column names, glued to a label
# ("day_7"). Safe, because it must also be >=3 columns increasing left to right.
_HEADER_NUMBER = re.compile(r"[-+]?\d*\.?\d+")
_ARTIFACT_HEADER = re.compile(
    r"^(unnamed[:_ ]|column[_ ]?\d+$|level[_ ]?\d+$|_duplicated_|index$)", re.IGNORECASE
)


def parse_axis_label(name: Any) -> Optional[float]:
    """Extract the ordinal position a column *name* denotes, if any."""
    text = str(name).strip().replace("−", "-")
    if not text or _ARTIFACT_HEADER.match(text):
        return None
    match = _HEADER_NUMBER.search(text)
    if not match:
        return None
    try:
        number = float(match.group())
    except ValueError:
        return None
    return number if math.isfinite(number) else None


# An ANOVA table spells its interaction block as one cell per row — "control, 6 days" is
# an axis point and an arm printed together. Splitting on separators is notation, the
# same kind of parsing as the trailing-unit rule: it finds *where* the number is, never
# what it measures.
_LABEL_PARTS = re.compile(r"\s*[,;|]\s*|\s+-\s+|\s+–\s+|\s+—\s+")


@lru_cache(maxsize=16384)
def _split_axis(text: str) -> tuple:
    parts = [part for part in _LABEL_PARTS.split(text) if part and part.strip()]
    if not parts:
        return None, text
    for index, part in enumerate(parts):
        position = parse_axis_label(part)
        if position is not None:
            residue = " ".join(other.strip() for at, other in enumerate(parts) if at != index)
            return position, residue.strip(" -_,")
    return None, text


def split_axis_from_label(label: Any) -> tuple:
    """Read an axis position out of a composite row label, and return what is left.

    ("control, 6 days") -> (6.0, "control"); a label carrying no position comes back
    unchanged beside None, so a caller can tell "not keyed" from "keyed with no arm".
    """
    return _split_axis(str(label or "").strip())


# α-terpinene and γ-terpinene are different compounds: folding the letter to ascii
# collapses every isomer in a GC-MS table onto one key, so it is spelled out instead.
_GREEK_TO_LATIN = str.maketrans({
    "α": "alpha", "β": "beta", "γ": "gamma", "δ": "delta", "ε": "epsilon",
    "ζ": "zeta", "η": "eta", "θ": "theta", "ι": "iota", "κ": "kappa",
    "λ": "lambda", "μ": "mu", "ν": "nu", "ξ": "xi", "ο": "omicron",
    "π": "pi", "ρ": "rho", "ς": "sigma", "σ": "sigma", "τ": "tau",
    "υ": "upsilon", "φ": "phi", "χ": "chi", "ψ": "psi", "ω": "omega",
})
_NON_ALNUM = re.compile(r"[^a-z0-9]+")
_UNDERSCORES = re.compile(r"_+")


@lru_cache(maxsize=32768)
def _canonical_key(text: str) -> str:
    folded = unicodedata.normalize("NFKD", text).lower().translate(_GREEK_TO_LATIN)
    folded = folded.encode("ascii", "ignore").decode("ascii")
    return _UNDERSCORES.sub("_", _NON_ALNUM.sub("_", folded)).strip("_")


def canonical_key(text: Any) -> str:
    """The comparison form of a label: ascii, lowercase, punctuation collapsed. Every
    "same term?" comparison goes through here, so two spellings match in one place, and
    it carries no vocabulary. Cached: it runs per label per observation."""
    return _canonical_key(str(text or ""))


# Silver appends this when a header repeats; stripping it keeps the unit readable.
DUPLICATE_SUFFIX = "__dup"
_DUPLICATE_TAIL = re.compile(re.escape(DUPLICATE_SUFFIX) + r"\d+$")

# "0.5% Moringa Extract", "Teo1%", "200 ppm nisin": this shape reveals a treatment arm
# rather than a measured quantity, without knowing what the substance is.
_DOSE = re.compile(
    r"""^\s*
    (?P<lead>.*?)\s*
    (?P<amount>\d+(?:[.,]\d+)?)\s*
    (?P<unit>%|ppm|ppb|mg/(?:g|ml|kg|l)|g/(?:kg|l)|µg/g|ug/g|mm|mg|w/w|v/v|w/v)
    \s*(?P<trail>.*?)\s*$""",
    re.VERBOSE | re.IGNORECASE,
)


class Dose(NamedTuple):
    substance: Optional[str]
    amount: float
    unit: str


@lru_cache(maxsize=8192)
def _parse_dosed(text: str) -> Optional[Dose]:
    match = _DOSE.match(_DUPLICATE_TAIL.sub("", text))
    if not match:
        return None
    lead, digits = match.group("lead"), match.group("amount")
    # "Se02%" is a 2% dose of something whose name ends in a letter the OCR read as a
    # zero — no one writes a dose with a leading zero. Give the digit back to the name.
    while len(digits) > 1 and digits[0] == "0" and "." not in digits:
        lead, digits = lead + "0", digits[1:]
    substance = " ".join(part for part in (lead, match.group("trail")) if part).strip(" -_,")
    return Dose(substance or None, float(digits.replace(",", ".")), match.group("unit").lower())


def parse_dosed_label(label: Any) -> Optional[Dose]:
    """Split "0.5% Moringa Extract" into substance, amount and unit, or None. Immutable
    and cached: the axis scorer, the wide fit, the arm reader and the validator all ask."""
    return _parse_dosed(str(label or "").strip())


# A trailing bracket carries the unit by typographic convention — "TVC (log CFU/g)".
# A formatting rule, so it stays domain-free: the gate never checks *which* unit.
_TRAILING_UNIT = re.compile(r"^(?P<name>.*?)[\s_]*[\(\[\{](?P<unit>[^\)\]\}]{1,24})[\)\]\}]\s*$")

# A header often carries two brackets — "APC (Log10 CFU/g) (mean ± Stdev)" — and the unit
# reader would take the last, which leaves the real unit inside the name and files the
# dispersion notation as the unit. Same notation the cell parser strips from a value, so
# it comes off here first.
_STATS_TAIL = re.compile(
    r"\s*[\(\[\{][^\)\]\}]*"
    r"(?:±|\bmean\b|\bmedian\b|\bstdev\b|\bstd\b|\bsd\b|\bsem?\b|\bn\s*=)"
    r"[^\)\]\}]*[\)\]\}]\s*$",
    re.IGNORECASE,
)


def split_label_and_unit(label: Any) -> tuple:
    """Split "TVC (log CFU/g)" into ("TVC", "log CFU/g"). Unit is None if absent."""
    text = _DUPLICATE_TAIL.sub("", str(label or "").strip()).strip()
    while True:
        trimmed = _STATS_TAIL.sub("", text).strip()
        if trimmed == text or not trimmed:
            break
        text = trimmed
    if not text:
        return "", None
    match = _TRAILING_UNIT.match(text)
    if not match:
        return text, None
    name = match.group("name").strip(" _-")
    unit = match.group("unit").strip()
    if not name or not unit:
        return text, None
    return name, unit


# ─── Results ──────────────────────────────────────────────────────────────────


@dataclass(slots=True)
class Observation:
    """One value at one axis point. `column_label` is the header it sat under and
    `row_labels` this row's entry per non-numeric column; they stay apart because which
    one names the indicator is a semantic question the gate does not answer."""

    axis_value: float
    value: float
    column_label: Optional[str] = None
    row_labels: dict = field(default_factory=dict)


@dataclass
class SchemaFit:
    fits: bool
    reason: str
    orientation: Optional[str] = None          # 'long' | 'wide'
    axis_label: Optional[str] = None
    axis_values: list = field(default_factory=list)
    axis_confidence: float = 0.0
    axis_runs: int = 0                         # how many times the axis restarts
    value_columns: list = field(default_factory=list)   # headers the values sat under
    label_columns: list = field(default_factory=list)   # non-numeric qualifier columns
    observations: list = field(default_factory=list)    # list[Observation]
    columns: list = field(default_factory=list, repr=False)   # the parsed _Column view

    def summary(self) -> dict:
        return {
            "fits": self.fits,
            "reason": self.reason,
            "orientation": self.orientation,
            "axis_label": self.axis_label,
            "axis_points": self.axis_values,
            "axis_confidence": round(self.axis_confidence, 2),
            "axis_runs": self.axis_runs,
            "value_columns": self.value_columns,
            "label_columns": self.label_columns,
            "observation_count": len(self.observations),
        }


# ─── Column view ──────────────────────────────────────────────────────────────


@dataclass
class _Column:
    """Set once in `_columns` and never mutated, so every derived statistic is cached:
    gating compares each column against every other, O(C^2) walks per table."""

    position: int
    name: str
    raw: list
    numeric: list                # list[float | None], aligned with raw

    @cached_property
    def values(self) -> list:
        return [value for value in self.numeric if value is not None]

    @cached_property
    def entries(self) -> list:
        """The non-empty cells, stripped — the denominator of every ratio below."""
        return [text for text in (("" if cell is None else str(cell).strip())
                                  for cell in self.raw) if text]

    @cached_property
    def numeric_ratio(self) -> float:
        return len(self.values) / len(self.entries) if self.entries else 0.0

    @cached_property
    def is_numeric(self) -> bool:
        return self.numeric_ratio >= MIN_NUMERIC_RATIO and len(self.values) >= MIN_AXIS_POINTS

    @cached_property
    def names_an_entity_per_row(self) -> bool:
        """A different entry on every row, over enough rows to be no coincidence."""
        return (len(self.entries) >= MIN_ENUMERATION_ROWS
                and len(set(self.entries)) == len(self.entries))

    @cached_property
    def keyed(self) -> list:
        """Per row, the axis position its label carries and what is left of the label.

        `[(6.0, 'control'), (None, 'treatments'), ...]`, aligned with `raw`. A block
        heading or a P-value row keys to None and drops out on its own.
        """
        return [split_axis_from_label(cell) if cell is not None else (None, "")
                for cell in self.raw]

    @cached_property
    def keys_an_axis(self) -> bool:
        """Enough rows carry a position, over enough distinct positions, to be an axis."""
        positions = [position for position, _ in self.keyed if position is not None]
        return (len(positions) >= MIN_SERIES_POINTS
                and len(set(positions)) >= MIN_AXIS_POINTS)


def _columns(headers: Sequence[Any], rows: Sequence[Sequence[Any]]) -> list:
    # Transposed in one pass rather than a bounds check per cell.
    transposed = list(zip_longest(*rows, fillvalue=None)) if rows else []
    result = []
    for index, name in enumerate(headers):
        raw = list(transposed[index]) if index < len(transposed) else [None] * len(rows)
        result.append(_Column(position=index, name=str(name), raw=raw,
                              numeric=[parse_number(cell) for cell in raw]))
    return result


def _row_labels(row: Sequence[Any], label_columns: Sequence[_Column]) -> dict:
    """This row's entry in each qualifier column. One str()/strip() per cell, not two."""
    labels = {}
    for column in label_columns:
        cell = row[column.position] if column.position < len(row) else None
        text = "" if cell is None else str(cell).strip()
        if text:
            labels[column.name] = text
    return labels


# ─── Axis detection (long orientation) ────────────────────────────────────────


def _runs(numeric: Sequence[Optional[float]]) -> list:
    """Maximal non-decreasing runs; a drop starts a new one. A time axis either climbs
    once or resets per group, and both are visible without knowing what it measures."""
    runs, current = [], []
    for value in numeric:
        if value is None:
            continue
        if current and value < current[-1] - EPS:
            runs.append(current)
            current = [value]
        else:
            current.append(value)
    if current:
        runs.append(current)
    return runs


def _enumerates_rows(column: _Column, columns: Sequence[_Column]) -> bool:
    """True when a column is just the row's position — "No.", "Item", "#" — which climbs
    perfectly while measuring nothing. Both conditions are needed: either alone would
    also catch a genuine daily time course."""
    positioned = [(index, value) for index, value in enumerate(column.numeric) if value is not None]
    if len(positioned) < MIN_ENUMERATION_ROWS:
        return False
    if len({value - index for index, value in positioned}) != 1:
        return False
    return any(other.names_an_entity_per_row for other in columns
               if other.position != column.position and not other.is_numeric)


def _axis_quality(column: _Column, columns: Sequence[_Column]) -> Optional[dict]:
    """Score a numeric column as an ordinal axis, or None if it cannot be one."""
    values = column.values
    if len(values) < MIN_AXIS_POINTS:
        return None
    if _enumerates_rows(column, columns):
        return None
    # A dosed header names a treatment arm, never an axis.
    if parse_dosed_label(column.name):
        return None

    runs = _runs(column.numeric)
    if not runs or min(len(set(run)) for run in runs) < MIN_AXIS_POINTS:
        return None

    distinct = sorted(set(values))
    if len(distinct) < MIN_AXIS_POINTS:
        return None

    # An axis repeats across groups; an indicator rarely does. Both signals structural.
    repeat_ratio = 1.0 - (len(distinct) / len(values))
    return {
        "runs": len(runs),
        "distinct": distinct,
        "repeat_ratio": repeat_ratio,
        "confidence": min(1.0, 0.4 + 0.3 * (len(runs) > 1) + 0.3 * repeat_ratio),
    }


def _fit_long(columns: list, rows: Sequence[Sequence[Any]],
              prefer_axis: Optional[str] = None) -> SchemaFit:
    numeric_columns = [column for column in columns if column.is_numeric]
    if not numeric_columns:
        return SchemaFit(fits=False, reason="no column holds 3 or more numeric values")

    candidates = []
    for column in numeric_columns:
        quality = _axis_quality(column, columns)
        if quality:
            candidates.append((column, quality))

    # A caller that already knows the paper's axis can name it. Chart OCR regularly
    # leaves one figure's time column slightly out of order, disqualifying it on its own
    # evidence while ten sibling figures agree on what the axis is.
    forced = None
    if prefer_axis:
        forced = next(
            (column for column in numeric_columns if canonical_key(column.name) == prefer_axis),
            None,
        )
    if forced is not None and not any(column is forced for column, _ in candidates):
        candidates.append((forced, {
            "runs": 0,
            "distinct": sorted(set(forced.values)),
            "repeat_ratio": 0.0,
            "confidence": 0.3,
        }))

    if not candidates:
        return SchemaFit(fits=False, reason="no column orders the rows like an axis")

    if forced is not None:
        column, quality = next(pair for pair in candidates if pair[0] is forced)
    else:
        # More resets beats better repetition beats leftmost. Repetition only counts when
        # the axis actually restarts: in a single-run column a repeated value is
        # coincidence, and letting it score put a treatment column that happened to hold
        # 3.9 twice ahead of the storage day column beside it.
        column, quality = max(
            candidates,
            key=lambda pair: (
                pair[1]["runs"],
                pair[1]["repeat_ratio"] if pair[1]["runs"] > 1 else 0.0,
                -pair[0].position,
            ),
        )

    value_columns = [
        other
        for other in numeric_columns
        if other.position != column.position
        and len(other.values) >= MIN_SERIES_POINTS
        and len(set(other.values)) >= 2
    ]
    if not value_columns:
        return SchemaFit(
            fits=False,
            reason="an axis but no varying measured column alongside it",
            orientation="long",
            axis_label=column.name,
            axis_values=quality["distinct"],
        )

    label_columns = [other for other in columns if not other.is_numeric and other.entries]

    observations = []
    axis_numeric = column.numeric
    for row_index, row in enumerate(rows):
        axis_value = axis_numeric[row_index]
        if axis_value is None:
            continue
        row_labels = _row_labels(row, label_columns)
        for measured in value_columns:
            value = measured.numeric[row_index]
            if value is not None:
                observations.append(
                    Observation(axis_value=axis_value, value=value,
                                column_label=measured.name, row_labels=row_labels)
                )

    return SchemaFit(
        fits=True,
        reason="long: one axis column, measured values in sibling columns",
        orientation="long",
        axis_label=column.name,
        axis_values=quality["distinct"],
        axis_confidence=quality["confidence"],
        axis_runs=quality["runs"],
        value_columns=[measured.name for measured in value_columns],
        label_columns=[label.name for label in label_columns],
        observations=observations,
    )


# ─── Axis detection (wide orientation) ────────────────────────────────────────


def _fit_wide(columns: list, rows: Sequence[Sequence[Any]]) -> SchemaFit:
    axis_columns = []
    for column in columns:
        position = parse_axis_label(column.name)
        if position is not None:
            axis_columns.append((position, column))

    if len(axis_columns) < MIN_AXIS_POINTS:
        return SchemaFit(fits=False, reason="fewer than 3 column names denote an axis position")

    positions = [position for position, _ in axis_columns]
    if any(later <= earlier for earlier, later in zip(positions, positions[1:])):
        return SchemaFit(fits=False, reason="axis-like column names do not increase left to right")

    # "0.5% Moringa extract | 1% | 2%" ascends beautifully and is not an axis: it is the
    # dose ladder of the study's arms. Reading such a table as wide swaps the treatment
    # onto the axis and buries the real time column among the labels.
    if sum(1 for _, column in axis_columns if parse_dosed_label(column.name)) >= 2:
        return SchemaFit(fits=False, reason="column names are a dose ladder, not an axis")

    parseable = sum(len(column.values) for _, column in axis_columns)
    total = sum(len(column.entries) for _, column in axis_columns)
    if not total or parseable / total < MIN_NUMERIC_RATIO:
        return SchemaFit(fits=False, reason="cells under the axis columns are not numeric")

    axis_positions = {column.position for _, column in axis_columns}
    label_columns = [column for column in columns
                     if column.position not in axis_positions and column.entries]

    observations = []
    rows_used = 0
    for row_index, row in enumerate(rows):
        points = [
            (position, column.numeric[row_index])
            for position, column in axis_columns
            if column.numeric[row_index] is not None
        ]
        if len(points) < MIN_SERIES_POINTS:
            continue
        rows_used += 1
        row_labels = _row_labels(row, label_columns) or {"row": f"row {row_index + 1}"}
        for position, value in points:
            observations.append(
                Observation(axis_value=position, value=value,
                            column_label=None,   # the axis owns the columns here
                            row_labels=row_labels)
            )

    if not rows_used:
        return SchemaFit(
            fits=False,
            reason="no row carries 3 or more points across the axis",
            orientation="wide",
        )

    return SchemaFit(
        fits=True,
        reason="wide: axis in the column names, one series per row",
        orientation="wide",
        axis_label="|".join(column.name for _, column in axis_columns),
        axis_values=positions,
        axis_confidence=0.9,
        axis_runs=1,
        value_columns=[column.name for _, column in axis_columns],
        label_columns=[label.name for label in label_columns],
        observations=observations,
    )


# ─── Axis detection (keyed orientation) ───────────────────────────────────────


#: The stub column an ANOVA table puts its row labels under is usually unheaded, and a
#: blank name is no use as a dictionary key — nor as something a reader can point at.
KEYED_LABEL = "row label"


def column_label(name: Any) -> str:
    """The name a column is known by. Blank headers are common on the stub column of an
    ANOVA table, and a column nobody can name is a column nobody can nominate as the
    axis."""
    return str(name or "").strip() or KEYED_LABEL


def _fit_keyed(columns: list, rows: Sequence[Sequence[Any]],
               prefer_axis: Optional[str] = None) -> SchemaFit:
    """The axis is inside a text label, one cell per row: "control, 6 days".

    How an ANOVA table spells its interaction block, and the house style of the corpus.
    The label's residue is carried as a row label under the column's own name, which is
    where the interpretation stage already looks for an arm.
    """
    candidates = [column for column in columns
                  if not column.is_numeric and column.keys_an_axis]
    if prefer_axis:
        named = [column for column in columns
                 if not column.is_numeric
                 and canonical_key(column_label(column.name)) == prefer_axis]
        candidates = named or candidates
    if not candidates:
        return SchemaFit(fits=False, reason="no label column carries an axis position")

    value_columns = [column for column in columns
                     if column.is_numeric and len(set(column.values)) >= 2]
    if not value_columns:
        return SchemaFit(fits=False, reason="a keyed axis but no varying measured column",
                         orientation="keyed")

    # Most distinct positions wins, leftmost breaks the tie — the same shape of rule the
    # long fit uses, and position is a layout convention used only as a tiebreak.
    column = max(candidates, key=lambda item: (
        len({position for position, _ in item.keyed if position is not None}), -item.position))

    keyed = column.keyed
    # An interaction block names an arm on every row; a marginal-means block does not.
    # When both are stacked in one table, the rows with an arm are the real series and
    # the rest are means *over* arms, which would otherwise become a phantom arm.
    arms_present = any(position is not None and residue for position, residue in keyed)

    other_labels = [item for item in columns
                    if item is not column and not item.is_numeric and item.entries]
    axis_name = column_label(column.name)
    observations = []
    used = set()
    for row_index, row in enumerate(rows):
        position, residue = keyed[row_index]
        if position is None or (arms_present and not residue):
            continue
        used.add(position)
        row_labels = _row_labels(row, other_labels)
        if residue:
            row_labels[axis_name] = residue
        for measured in value_columns:
            value = measured.numeric[row_index]
            if value is not None:
                observations.append(
                    Observation(axis_value=position, value=value,
                                column_label=measured.name, row_labels=row_labels)
                )

    if len(used) < MIN_AXIS_POINTS:
        return SchemaFit(fits=False, reason="fewer than 3 axis positions survive keying",
                         orientation="keyed")

    positions = [position for position, _ in keyed if position is not None]
    return SchemaFit(
        fits=True,
        reason="keyed: the axis is inside a row label, measured values alongside",
        orientation="keyed",
        axis_label=axis_name,
        axis_values=sorted(used),
        axis_confidence=0.7,
        axis_runs=len(_runs(positions)),
        value_columns=[measured.name for measured in value_columns],
        label_columns=[item.name for item in other_labels] + [axis_name],
        observations=observations,
    )


# ─── Public table gate ────────────────────────────────────────────────────────


def fits_schema(headers: Sequence[Any], rows: Sequence[Sequence[Any]],
                prefer_axis: Optional[str] = None) -> SchemaFit:
    """Does this table hold values observed over an ordinal axis?

    Every orientation is fitted and the stronger evidence wins: a long axis that
    *restarts* beats everything, then an axis spelled out in the column names, then a
    long axis that climbs once, then an axis keyed out of a row label. Wide cannot go
    first — a dose ladder in the header reads as a perfectly good ascending axis while
    the real time column is ignored — and keyed goes last, because reading a number out
    of prose is the weakest evidence here. The parsed columns ride along so the reference
    test need not re-parse a rejected table.
    """
    if not headers or not rows:
        return SchemaFit(fits=False, reason="empty table")

    columns = _columns(headers, rows)

    # A hint naming a text column can only be answered by the keyed fit, so it goes
    # straight there rather than losing to a long fit that ignored the hint.
    if prefer_axis and any(not column.is_numeric
                           and canonical_key(column_label(column.name)) == prefer_axis
                           for column in columns):
        fit = _fit_keyed(columns, rows, prefer_axis=prefer_axis)
        fit.columns = columns
        return fit

    long = _fit_long(columns, rows, prefer_axis=prefer_axis)
    wide = keyed = None
    if not (long.fits and (long.axis_runs > 1 or prefer_axis)):
        wide = _fit_wide(columns, rows)
    if not (long.fits or (wide is not None and wide.fits)):
        keyed = _fit_keyed(columns, rows, prefer_axis=prefer_axis)

    if wide is not None and wide.fits:
        fit = wide
    elif long.fits:
        fit = long
    elif keyed is not None and keyed.fits:
        fit = keyed
    # Report whichever attempt got further, so the rejection is explainable. `wide` only
    # sets an orientation once it has accepted an axis, so when it did, its complaint is
    # the specific one; otherwise `long` has the better story.
    elif wide is not None and wide.orientation:
        fit = wide
    else:
        fit = long

    fit.columns = columns
    return fit


# ─── Figure pre-filter ────────────────────────────────────────────────────────
# Geometry and colour only: a plot is a drawing on a uniform ground with a long straight
# rule in each direction, a photograph has no uniform ground, a logo is too small.

MIN_SIDE_PX = 120
MIN_AREA_PX = 40_000
ASPECT_BOUNDS = (0.2, 5.0)
MIN_BACKGROUND_FRACTION = 0.35
MAX_INK_FRACTION = 0.60
MIN_INK_FRACTION = 0.004
MIN_RULE_SPAN = 0.55        # a rule must cross 55% of the frame
PROBE_LONG_SIDE = 320
INK_THRESHOLD = 32          # per-channel distance from the ground colour


@dataclass
class PlotProbe:
    plot_like: bool
    reason: str
    width: int = 0
    height: int = 0
    background_fraction: float = 0.0
    ink_fraction: float = 0.0
    distinct_colours: int = 0
    h_rule: float = 0.0
    v_rule: float = 0.0

    def summary(self) -> dict:
        return {
            "plot_like": self.plot_like,
            "reason": self.reason,
            "width": self.width,
            "height": self.height,
            "background_fraction": round(self.background_fraction, 3),
            "ink_fraction": round(self.ink_fraction, 3),
            "distinct_colours": self.distinct_colours,
            "h_rule": round(self.h_rule, 3),
            "v_rule": round(self.v_rule, 3),
        }


def _rule_span(mask, axis: int) -> float:
    """How far the strongest straight rule reaches, as a fraction of the frame. Either an
    unbroken run or a line mostly inked along its length, so a dashed or tick-marked
    spine counts without admitting scattered ink."""
    grid = mask if axis == 1 else mask.T
    span = grid.shape[1]
    if span == 0:
        return 0.0

    pad = np.zeros((grid.shape[0], 1), dtype=bool)
    padded = np.concatenate([pad, grid, pad], axis=1).astype(np.int8)
    deltas = np.diff(padded, axis=1)
    _, starts = np.where(deltas == 1)
    _, ends = np.where(deltas == -1)
    longest_run = int((ends - starts).max()) if starts.size else 0

    return max(longest_run / span, float(grid.mean(axis=1).max()))


def plot_likeness(image_path: str) -> PlotProbe:
    """Cheap, content-free test for 'this image could be a data display'."""
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as exc:
        return PlotProbe(plot_like=False, reason=f"unreadable image ({exc})")

    width, height = image.size
    aspect = width / height if height else 0.0
    if not (ASPECT_BOUNDS[0] <= aspect <= ASPECT_BOUNDS[1]):
        return PlotProbe(False, f"aspect ratio {aspect:.2f} outside plot range", width, height)

    if min(width, height) < MIN_SIDE_PX or width * height < MIN_AREA_PX:
        return PlotProbe(False, "too small to be a data display", width, height)

    scale = PROBE_LONG_SIDE / max(width, height)
    if scale < 1.0:
        image = image.resize(
            (max(1, int(width * scale)), max(1, int(height * scale))), Image.BILINEAR
        )
    pixels = np.asarray(image, dtype=np.int16)

    # Ground colour = the most common colour after quantisation.
    quantised = (pixels // 32).astype(np.uint8)
    codes = quantised[..., 0] * 64 + quantised[..., 1] * 8 + quantised[..., 2]
    counts = np.bincount(codes.ravel(), minlength=512)
    ground_code = int(counts.argmax())
    background_fraction = float(counts.max()) / codes.size

    ground = pixels[codes == ground_code].mean(axis=0)
    ink = (np.abs(pixels - ground).max(axis=2) > INK_THRESHOLD)
    ink_fraction = float(ink.mean())

    probe = PlotProbe(
        plot_like=False,
        reason="",
        width=width,
        height=height,
        background_fraction=background_fraction,
        ink_fraction=ink_fraction,
        distinct_colours=int((counts > 0).sum()),
    )

    if background_fraction < MIN_BACKGROUND_FRACTION:
        probe.reason = "no uniform ground — photographic or dense imagery"
        return probe
    if ink_fraction > MAX_INK_FRACTION:
        probe.reason = "image is mostly ink"
        return probe
    if ink_fraction < MIN_INK_FRACTION:
        probe.reason = "image is effectively blank"
        return probe

    probe.h_rule = _rule_span(ink, axis=1)
    probe.v_rule = _rule_span(ink, axis=0)
    if probe.h_rule < MIN_RULE_SPAN or probe.v_rule < MIN_RULE_SPAN:
        probe.reason = "no axis-like rules in both directions"
        return probe

    probe.plot_like = True
    probe.reason = "uniform ground with rules on both axes"
    return probe

## Silver

Deterministic cleaning, then the gate. Output is a package of gated assets with their observations parsed into (axis, label, value) tuples.

In [4]:
import polars as pl

# ─── Table cleaning ───────────────────────────────────────────────────────────
# Layout damage only: whitespace, empty rows and columns, a header in the first data row,
# a footnote in the last. Column names are never rewritten — the unit and the wide axis
# both live in the header, so renaming would throw away what the gate reads.

_POSITIONAL_HEADER = re.compile(r"^(unnamed[:_ ]|column[_ ]?\d+$|\d+$|_duplicated_)", re.IGNORECASE)


def dedupe_names(names: Sequence[Any]) -> list:
    """Make column names unique, keeping the first occurrence untouched: a two-row header
    repeats its group label, so lifting it collapses two columns onto one name."""
    seen: dict = {}
    unique = []
    for name in names:
        text = str(name)
        seen[text] = seen.get(text, 0) + 1
        unique.append(text if seen[text] == 1 else f"{text}{DUPLICATE_SUFFIX}{seen[text]}")
    return unique


def promote_header_row(frame: pl.DataFrame) -> pl.DataFrame:
    """Lift row 0 into the header when Docling left the columns positional."""
    if frame.height < 2:
        return frame
    threshold = max(1, int(0.6 * frame.width))
    if sum(1 for name in frame.columns if _POSITIONAL_HEADER.match(str(name))) < threshold:
        return frame
    first = [str(value).strip() if value is not None else "" for value in frame.row(0)]
    if sum(1 for value in first if value) < threshold:
        return frame

    promoted = frame.slice(1)
    # Assign the names rather than `rename`: a mapping cannot express a repeated target,
    # and polars rejects the collapse outright.
    promoted.columns = dedupe_names(new or old for old, new in zip(frame.columns, first))
    return promoted


def drop_footnote_rows(frame: pl.DataFrame) -> pl.DataFrame:
    """Drop trailing rows holding one long free-text cell — the shape of a footnote."""
    height = frame.height
    while height:
        filled = [str(value).strip() for value in frame.row(height - 1)
                  if value is not None and str(value).strip()]
        if len(filled) == 1 and len(filled[0]) > 40:
            height -= 1
            continue
        break
    return frame if height == frame.height else frame.slice(0, height)


def clean_table_frame(frame: pl.DataFrame) -> pl.DataFrame:
    frame = promote_header_row(frame)
    frame.columns = dedupe_names(frame.columns)
    # One with_columns for every text column, not one new frame per column.
    stripped = [pl.col(name).str.strip_chars()
                for name, dtype in zip(frame.columns, frame.dtypes) if dtype == pl.Utf8]
    if stripped:
        frame = frame.with_columns(stripped)
    frame = frame.select([name for name in frame.columns if not frame[name].is_null().all()])
    if frame.height and frame.width:
        frame = frame.filter(pl.any_horizontal([pl.col(c).is_not_null() for c in frame.columns]))
    return drop_footnote_rows(frame)


def frame_to_records(frame: pl.DataFrame) -> tuple:
    """The (headers, rows) pair the gate consumes."""
    return list(frame.columns), [list(row) for row in frame.iter_rows()]


def as_markdown_table(headers: Sequence[Any], rows: Sequence[Sequence[Any]], limit: int = 5) -> str:
    """A small pipe table for prompts and previews, without pulling in tabulate."""
    def cell(value):
        return "" if value is None else str(value).replace("|", "\\|").strip()

    head = [cell(h) for h in headers]
    lines = ["| " + " | ".join(head) + " |", "| " + " | ".join("---" for _ in head) + " |"]
    for row in rows[:limit]:
        lines.append("| " + " | ".join(cell(value) for value in row) + " |")
    if len(rows) > limit:
        lines.append(f"_({len(rows) - limit} more rows)_")
    return "\n".join(lines)


# ─── Sections and per-figure context ──────────────────────────────────────────

def markdown_sections(markdown_text: str) -> list:
    sections = []
    current_title = "Document"
    current_lines = []
    for line in markdown_text.splitlines():
        heading = re.match(r"^(#{1,6})\s+(.*)$", line.strip())
        if heading:
            if current_lines:
                sections.append({
                    "section_title": current_title,
                    "content_markdown": "\n".join(current_lines).strip(),
                })
            current_title = heading.group(2).strip()
            current_lines = []
        else:
            current_lines.append(line)
    if current_lines:
        sections.append({
            "section_title": current_title,
            "content_markdown": "\n".join(current_lines).strip(),
        })
    # A ref per section, assigned after the empties are dropped so it stays contiguous.
    # Prose is where a protocol and a sample mass are stated, so without one the two
    # fields that come only from prose would have nothing citable to point at.
    kept = [section for section in sections if section["content_markdown"]]
    for order, section in enumerate(kept):
        section["docling_item_ref"] = f"#/sections/{order}"
    return kept


def build_text_index(texts: Sequence[dict]) -> dict:
    """Bucket the paper's text items by page, once, for every asset to share."""
    by_page: dict = {}
    for item in texts:
        by_page.setdefault(item.get("page_number"), []).append(item)
    return {"all": list(texts), "by_page": by_page}


def local_context(text_index: dict, anchor: dict, budget: int = FIGURE_CONTEXT_CHARS) -> dict:
    """Text near an asset, chosen by position rather than by words. The page is the
    tightest locality the document gives us, and reading order is the fallback for a page
    that is all figure."""
    order = anchor.get("order", 0)
    page = anchor.get("page_number")
    candidates = text_index["by_page"].get(page) if page is not None else None
    if not candidates:
        candidates = text_index["all"]

    ranked = sorted(candidates, key=lambda item: abs(item.get("order", 0) - order))
    chosen, used = [], 0
    for item in ranked:
        text = item.get("text", "")
        if used + len(text) > budget and chosen:
            break
        chosen.append(item)
        used += len(text)
    chosen.sort(key=lambda item: item.get("order", 0))
    headings = [item.get("heading") for item in chosen if item.get("heading")]
    return {
        "context_markdown": "\n\n".join(item["text"] for item in chosen),
        "context_refs": [item["docling_item_ref"] for item in chosen],
        "section_hint": headings[0] if headings else None,
    }


# ─── Figure → table conversion ────────────────────────────────────────────────
# A figure is judged on the data recovered from it, so it must be converted first. The
# probe in the gate cell keeps that cost off photographs and logos.

_chart_models: dict = {}


def get_chart_model(device=None):
    """Load PP-Chart2Table once per device; a model that cannot load raises rather than
    degrading the run into "this paper has no plottable figures"."""
    device = device or ("gpu" if CUDA_AVAILABLE else "cpu")
    if device in _chart_models:
        return _chart_models[device]
    try:
        from paddleocr import ChartParsing
    except ImportError as exc:
        raise ImportError(
            "paddleocr with ChartParsing is required to test figures against the "
            "schema. Install it, or set REQUIRE_FIGURE_DATA=0 to admit figures on "
            "the visual probe alone."
        ) from exc
    # engine="transformers": paddle's native inference backend raises "Type of attribute:
    # strides is not right" on some builds, and this is the supported way around it.
    _chart_models[device] = ChartParsing(
        model_name="PP-Chart2Table",
        engine="transformers",
        engine_config={"dtype": "float16" if device == "gpu" else "float32"},
        device=device,
    )
    print(f"PP-Chart2Table loaded on {device}.")
    return _chart_models[device]


def parse_markdown_table(text: str) -> Optional[tuple]:
    """Parse the pipe table PP-Chart2Table returns into (headers, rows)."""
    if not text or not text.strip():
        return None
    lines = [line for line in text.splitlines() if line.strip()]
    lines = [line for line in lines if not re.match(r"^\s*\|?[\s\-:]+(\|[\s\-:]+)*\|?\s*$", line)]
    if len(lines) < 2:
        return None
    grid = []
    for line in lines:
        cells = [cell.strip() for cell in line.strip().strip("|").split("|")]
        if any(cells):
            grid.append(cells)
    if len(grid) < 2:
        return None
    width = max(len(row) for row in grid)
    grid = [row + [None] * (width - len(row)) for row in grid]
    # Two series sharing a legend label convert to two identically named columns; the
    # gate reads columns positionally but the CSV cannot.
    return dedupe_names(grid[0]), grid[1:]


def figure_result(status: str, **extra) -> dict:
    """One shape for every conversion outcome, so callers never branch on a key."""
    return {"status": status, "headers": None, "rows": None, **extra}


def _free_cuda() -> None:
    """`empty_cache()` only returns blocks nothing references, and the tensors of a failed
    forward pass are still held by the traceback of the exception reporting it — so the
    collection has to come first or almost nothing is given back."""
    gc.collect()
    if torch is not None and CUDA_AVAILABLE:
        torch.cuda.empty_cache()


def release_docling_models() -> bool:
    """Drop the cached converters. `empty_cache()` never releases model weights, so
    Docling's layout and table models sit on the card for the whole session — which is
    most of what the chart model is competing with. Only our references go, so a version
    holding its own singletons may keep some; the next Bronze rebuilds the converter."""
    if not _CONVERTER_CACHE:
        return False
    _CONVERTER_CACHE.clear()
    _free_cuda()
    print("        released Docling's models to make room for the chart model")
    return True


def _model_inputs(image_paths: Sequence[str], max_side: int, workdir: Path,
                  offset: int = 0) -> list:
    """Bounded copies of the figures, positionally aligned with `image_paths`.

    The model tiles what it is given, so its cost follows the pixel count rather than
    anything on disk; the archived PNG keeps its full IMAGES_SCALE resolution.
    """
    prepared = []
    for index, path in enumerate(image_paths, start=offset):
        try:
            with Image.open(path) as opened:
                opened.load()
                if max(opened.size) <= max_side:
                    prepared.append(str(path))
                    continue
                scale = max_side / max(opened.size)
                resized = opened.convert("RGB").resize(
                    (max(1, int(opened.width * scale)), max(1, int(opened.height * scale))),
                    Image.LANCZOS)
            target = workdir / f"chart_{index:03d}_{max_side}.png"
            resized.save(target)
            prepared.append(str(target))
        except (OSError, ValueError):
            prepared.append(str(path))   # unreadable here is unreadable for the model too
    return prepared


def _is_out_of_memory(exc: BaseException) -> bool:
    """Torch raises a dedicated class on newer versions and a plain RuntimeError on older
    ones; the message is the only thing common to both."""
    dedicated = getattr(torch, "cuda", None) and getattr(torch.cuda, "OutOfMemoryError", None)
    return (dedicated is not None and isinstance(exc, dedicated)) or (
        isinstance(exc, RuntimeError) and "out of memory" in str(exc).lower())


def _convert_chunk(model, chunk: Sequence[str]) -> list:
    """One model call. Results are positional, so a short list would silently discard the
    tail — hence the length check rather than a forgiving zip."""
    predictions = list(model.predict(
        input=[{"image": str(path)} for path in chunk], batch_size=len(chunk)
    ))
    if len(predictions) != len(chunk):
        raise RuntimeError(
            f"PP-Chart2Table returned {len(predictions)} predictions for {len(chunk)} "
            f"figures; results are positional, so the mismatch cannot be attributed "
            f"and the paper is not partially converted."
        )
    results = []
    for prediction in predictions:
        parsed = parse_markdown_table(prediction.get("result", ""))
        results.append(figure_result("converted", headers=parsed[0], rows=parsed[1])
                       if parsed else figure_result("unparseable"))
    return results


def convert_figures(image_paths: Sequence[str]) -> list:
    """Recover a table from each chart image, in as few model calls as fit in memory.

    Batched because model invocation, not the images, dominates the cost. Everything
    below is what happens when a batch does not fit; see `_convert_within_memory`.
    """
    if not image_paths:
        return []
    if not REQUIRE_FIGURE_DATA:
        # Admitted on the visual probe alone, so the model is never loaded.
        return [figure_result("unavailable") for _ in image_paths]

    with tempfile.TemporaryDirectory(prefix="chart_inputs_") as scratch:
        return _convert_within_memory(image_paths, Path(scratch))


def _convert_within_memory(image_paths: Sequence[str], scratch: Path) -> list:
    """Convert every figure, giving up something at each out-of-memory in turn.

    In order, because each step costs more than the one before it: halve the batch, hand
    back Docling's resident models, show the model a smaller copy, move to the CPU. Only
    a figure that survives all four comes back as `oom`, and it costs that figure rather
    than the paper — one unconvertible chart is not a reason to lose the other twelve.
    """
    device = "gpu" if CUDA_AVAILABLE else "cpu"
    max_side = CHART_MAX_PIXELS
    prepared = _model_inputs(image_paths, max_side, scratch)
    model = get_chart_model(device)
    results, batch, index, freed_docling = [], max(1, CHART_BATCH_SIZE), 0, False

    while index < len(prepared):
        chunk = prepared[index:index + batch]
        try:
            results.extend(_convert_chunk(model, chunk))
        except Exception as exc:
            if not _is_out_of_memory(exc):
                raise
            del exc          # its traceback pins every tensor of the failed pass
            _free_cuda()

            if batch > 1:
                batch = max(1, batch // 2)
                print(f"        chart conversion out of memory; batch -> {batch}")
            elif not freed_docling and release_docling_models():
                freed_docling = True
            elif max_side > CHART_MIN_PIXELS:
                max_side = max(CHART_MIN_PIXELS, max_side // 2)
                prepared[index:] = _model_inputs(image_paths[index:], max_side, scratch, index)
                print(f"        still out of memory; showing the model {max_side} px")
            elif device == "gpu" and CHART_CPU_FALLBACK:
                device = "cpu"
                _chart_models.pop("gpu", None)
                _free_cuda()
                model = get_chart_model(device)
                # RAM is the constraint now, so the figures go back to full size.
                max_side = CHART_MAX_PIXELS
                prepared[index:] = _model_inputs(image_paths[index:], max_side, scratch, index)
                print("        no GPU memory for one figure; converting on the CPU (slow)")
            else:
                results.append(figure_result("oom"))
                index += 1
                print(f"        figure {index} does not fit in memory — skipped")
            continue
        index += len(chunk)
        _free_cuda()
    return results


# ─── Silver build ─────────────────────────────────────────────────────────────

def _observation_payload(fit: SchemaFit) -> list:
    """Serialise observations with their labels still separate."""
    return [
        {
            "axis_value": observation.axis_value,
            "value": observation.value,
            "column_label": observation.column_label,
            "row_labels": observation.row_labels,
        }
        for observation in fit.observations
    ]


def _asset_base(meta: dict, kind: str) -> dict:
    return {
        "kind": kind,
        "index": meta.get("table_index") if kind == "table" else meta.get("figure_index"),
        "docling_item_ref": meta.get("docling_item_ref"),
        "caption": meta.get("caption"),
        "page_number": meta.get("page_number"),
    }


MIN_REFERENCE_ENTITIES = 3
#: Rows of a review asset the model is shown. Enough to see the shape of the first block
#: of an ANOVA table, not so many that a dozen assets crowd out the methods prose.
REVIEW_PREVIEW_ROWS = int(os.getenv("REVIEW_PREVIEW_ROWS", "10"))


def _write_figure_csv(headers: Sequence[Any], rows: Sequence[Sequence[Any]], path: Path) -> None:
    pl.DataFrame(
        {name: [row[i] if i < len(row) else None for row in rows]
         for i, name in enumerate(headers)},
        strict=False,
    ).write_csv(path)


def _is_reference_table(fit: SchemaFit, row_count: int) -> bool:
    """A catalogue of entities against quantities rather than a series over time. A GC-MS
    breakdown has no axis, so the gate rejects it, but its rows are *substances*: a column
    of distinct names beside a column of numbers, read off the gate's parsed columns.

    A column whose labels key an axis is a series that was not read, not a catalogue —
    without that test an ANOVA table's "control, 6 days" column reads as a list of
    substance names and its rows are mined into `ingredients`.
    """
    if not fit.columns or row_count < MIN_REFERENCE_ENTITIES:
        return False
    names = numbers = False
    for column in fit.columns:
        if not column.entries:
            continue
        if column.numeric_ratio >= MIN_NUMERIC_RATIO:
            numbers = True
        elif column.keys_an_axis:
            return False
        elif len(set(column.entries)) >= MIN_REFERENCE_ENTITIES:
            names = True
    return names and numbers


def _needs_review(fit: SchemaFit) -> bool:
    """Shaped like data, but the gate could not key it.

    At least one column of numbers and at least one column of labels, and no axis in
    either — which is a measurement table the gate has not understood far more often than
    it is furniture. Sending these to the model to have their axis named beats dropping
    them into `rejected` beside the logos.
    """
    if not fit.columns:
        return False
    return (any(column.is_numeric for column in fit.columns)
            and any(not column.is_numeric and column.entries for column in fit.columns))


def gate_table(table_meta: dict, text_index: dict, tables_dir: Path) -> tuple:
    """Clean, gate and stage one native table. Returns (verdict, record)."""
    base = _asset_base(table_meta, "table")
    csv_path = table_meta.get("csv_path")
    if not csv_path:
        return "rejected", {**base, "stage": "bronze", "reason": table_meta.get("error", "no CSV")}

    frame = clean_table_frame(pl.read_csv(csv_path, infer_schema_length=50, ignore_errors=True))
    headers, rows = frame_to_records(frame)
    fit = fits_schema(headers, rows)
    cleaned_csv_path = tables_dir / Path(csv_path).name

    if not fit.fits:
        if _is_reference_table(fit, len(rows)):
            frame.write_csv(cleaned_csv_path)
            return "reference", {
                **table_meta,
                "cleaned_csv_path": str(cleaned_csv_path),
                "headers": headers,
                "rows": rows,
                "gate": fit.summary(),
                "why": f"no measurement axis ({fit.reason}); rows name entities",
                **local_context(text_index, table_meta),
            }
        if _needs_review(fit):
            frame.write_csv(cleaned_csv_path)
            return "review", {
                **table_meta,
                "is_figure": False,
                "cleaned_csv_path": str(cleaned_csv_path),
                "headers": headers,
                "rows": rows,
                "preview_markdown": as_markdown_table(headers, rows, limit=REVIEW_PREVIEW_ROWS),
                "gate": fit.summary(),
                "why": fit.reason,
                **local_context(text_index, table_meta),
            }
        return "rejected", {**base, "stage": "schema_gate", "reason": fit.reason}

    frame.write_csv(cleaned_csv_path)
    return "accepted", {
        **table_meta,
        "is_figure": False,
        "cleaned_csv_path": str(cleaned_csv_path),
        "headers": headers,
        "row_count": frame.height,
        "col_count": frame.width,
        "preview_markdown": as_markdown_table(headers, rows),
        "gate": fit.summary(),
        "observations": _observation_payload(fit),
        **local_context(text_index, table_meta),
    }


def probe_figure(figure_meta: dict) -> tuple:
    """Cheap visual test, no model. Split from the gate so every surviving figure can be
    converted in one batched call."""
    base = _asset_base(figure_meta, "figure")
    if not figure_meta.get("image_path"):
        return "rejected", {**base, "stage": "bronze", "reason": figure_meta.get("error", "no image")}
    probe = plot_likeness(figure_meta["image_path"])
    if not probe.plot_like:
        return "rejected", {**base, "stage": "probe", "reason": probe.reason,
                            "probe": probe.summary()}
    return "probed", probe


def gate_converted_figure(figure_meta: dict, probe, conversion: dict,
                          text_index: dict, figures_dir: Path) -> tuple:
    """Gate one already-converted figure. Returns (verdict, record)."""
    base = _asset_base(figure_meta, "figure")
    if conversion["status"] != "converted":
        if conversion["status"] == "unavailable" and not REQUIRE_FIGURE_DATA:
            return "accepted", {
                **figure_meta,
                "is_figure": True,
                "probe": probe.summary(),
                "conversion_status": "unavailable",
                "gate": {"fits": None, "reason": "admitted on the visual probe alone"},
                "observations": [],
                **local_context(text_index, figure_meta),
            }
        return "rejected", {
            **base,
            "stage": "conversion",
            "reason": conversion.get("error") or conversion["status"],
            "probe": probe.summary(),
        }

    headers, rows = conversion["headers"], conversion["rows"]
    fit = fits_schema(headers, rows)
    figure_csv_path = figures_dir / f"figure_{figure_meta['figure_index']:03d}.csv"
    if not fit.fits:
        if not _needs_review(fit):
            return "rejected", {**base, "stage": "schema_gate", "reason": fit.reason,
                                "probe": probe.summary()}
        _write_figure_csv(headers, rows, figure_csv_path)
        return "review", {
            **figure_meta,
            "is_figure": True,
            "probe": probe.summary(),
            "conversion_status": "converted",
            "csv_path": str(figure_csv_path),
            "headers": headers,
            "rows": rows,
            "preview_markdown": as_markdown_table(headers, rows, limit=REVIEW_PREVIEW_ROWS),
            "gate": fit.summary(),
            "why": fit.reason,
            **local_context(text_index, figure_meta),
        }

    _write_figure_csv(headers, rows, figure_csv_path)

    return "accepted", {
        **figure_meta,
        "is_figure": True,
        "probe": probe.summary(),
        "conversion_status": "converted",
        "csv_path": str(figure_csv_path),
        "headers": headers,
        "preview_markdown": as_markdown_table(headers, rows),
        "gate": fit.summary(),
        "observations": _observation_payload(fit),
        **local_context(text_index, figure_meta),
    }


# One malformed asset must not cost the paper. These are the failures a bad asset
# actually produces; anything else is a bug and is left to propagate.
ASSET_FAILURES = (ValueError, TypeError, KeyError, IndexError, OSError, pl.exceptions.PolarsError)


def build_silver_package(manifest: dict) -> dict:
    slug = manifest["paper_slug"]
    silver_dir = SILVER_ROOT / slug
    if silver_dir.exists():
        shutil.rmtree(silver_dir)
    tables_dir = silver_dir / "tables"
    figures_dir = silver_dir / "figures"
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    sections = markdown_sections(Path(manifest["markdown_path"]).read_text(encoding="utf-8"))
    text_index = build_text_index(manifest.get("texts", []))
    table_metas = manifest.get("tables", [])
    figure_metas = manifest.get("figures", [])

    accepted_tables, accepted_figures, rejected, references, review = [], [], [], [], []
    buckets = {"reference": references, "review": review, "rejected": rejected}

    def dispatch(accepted, verdict, record):
        (accepted if verdict == "accepted" else buckets[verdict]).append(record)

    def rejection(meta, kind, exc):
        return {**_asset_base(meta, kind), "stage": "error",
                "reason": f"{type(exc).__name__}: {exc}"}

    # 1. Tables, with no model in the loop at all.
    for meta in table_metas:
        try:
            dispatch(accepted_tables, *gate_table(meta, text_index, tables_dir))
        except ASSET_FAILURES as exc:
            rejected.append(rejection(meta, "table", exc))

    # 2. Probe every figure — cheap, and it keeps the model away from photographs.
    probed = []
    for meta in figure_metas:
        try:
            verdict, payload = probe_figure(meta)
        except ASSET_FAILURES as exc:
            rejected.append(rejection(meta, "figure", exc))
            continue
        if verdict == "probed":
            probed.append((meta, payload))
        else:
            rejected.append(payload)

    # 3. One model call for everything that survived; paths key the results positionally.
    conversions = convert_figures([meta["image_path"] for meta, _ in probed])

    # 4. Gate each converted figure.
    for (meta, probe), conversion in zip(probed, conversions):
        try:
            dispatch(accepted_figures,
                     *gate_converted_figure(meta, probe, conversion, text_index, figures_dir))
        except ASSET_FAILURES as exc:
            rejected.append(rejection(meta, "figure", exc))

    gate_report = {
        "tables_in": len(table_metas),
        "tables_accepted": len(accepted_tables),
        "figures_in": len(figure_metas),
        "figures_accepted": len(accepted_figures),
        "rejected_by_stage": {
            stage: sum(1 for item in rejected if item["stage"] == stage)
            for stage in sorted({item["stage"] for item in rejected})
        },
        "references": len(references),
        "review": len(review),
        "observations": sum(len(item["observations"])
                            for item in chain(accepted_tables, accepted_figures)),
    }

    silver_manifest = {
        "paper_slug": slug,
        "source_pdf": manifest["source_pdf"],
        "file_hash": manifest["file_hash"],
        "sections": sections,
        "tables": accepted_tables,
        "figures": accepted_figures,
        "references": references,
        "review": review,
        "rejected": rejected,
        "gate_report": gate_report,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    (silver_dir / "manifest.json").write_text(
        json.dumps(silver_manifest, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
    )
    print(f"Silver: {slug} | tables {gate_report['tables_accepted']}/{gate_report['tables_in']} "
          f"| figures {gate_report['figures_accepted']}/{gate_report['figures_in']} "
          f"| {gate_report['observations']} observations "
          f"| {gate_report['references']} reference | {gate_report['review']} for review "
          f"| rejected {gate_report['rejected_by_stage']}")
    return silver_manifest


# ─── Review adjudication ──────────────────────────────────────────────────────
# The gate found numbers and labels but could not say which label orders the rows. The
# model is asked that question and nothing else: it names a column, and the gate re-reads
# the table itself. No value it returns could reach the database, because it is not asked
# for any — which is the same contract Gold keeps for the two fields it does supply.
# `AssetHint` validates the reply and lives with the other schemas in the Gold cell,
# which has run by the time anything here is called.

REVIEW_SYSTEM_PROMPT = """\
You are reading tables and charts recovered from one scientific paper.

Each asset below has numbers in it, but the software could not tell which column orders
the rows into a series. Name that column. Do NOT transcribe any values.

For each asset return one object:

{
  "docling_item_ref": REQUIRED - copy it from the asset,
  "axis_column": the exact name, copied from this asset's `columns`, of the column whose
                 cells carry the point each row was observed at - a storage day, a
                 concentration, a time. The cell may hold more than that, as in
                 "control, 6 days"; name it anyway. Use null if no column does.
  "not_a_series": true when this is not repeated measurements at all - a composition
                  breakdown, a list of equipment, a statistics summary - else false,
  "why": one short sentence
}

Rules:
  * `axis_column` must be copied verbatim from that asset's `columns`, or be null. An
    invented name is discarded. An unheaded column is listed as "row label"; that is a
    name you may use.
  * A column of measured quantities is NOT the axis. The axis is what those quantities
    were measured against, and it is a column of text here - if it were a column of plain
    numbers the software would already have found it.
  * If several columns could be it, pick the one whose values repeat across groups.

Return ONLY valid JSON: {"assets": [ ... ]}
"""


def _review_prompt(package: dict) -> str:
    return json.dumps({
        "paper_slug": package["paper_slug"],
        "assets": [{
            "docling_item_ref": asset.get("docling_item_ref"),
            "kind": "figure" if asset.get("is_figure") else "table",
            "caption": asset.get("caption"),
            "columns": [column_label(name) for name in asset.get("headers", [])],
            "why_unresolved": asset.get("why"),
            "preview": asset.get("preview_markdown"),
        } for asset in package.get("review", [])],
    }, ensure_ascii=False, indent=2, default=str)


def adjudicate_review(package: dict, client) -> int:
    """Ask the model which column is the axis, then re-gate with its answer.

    Runs in Silver, where structure is resolved, and returns how many assets it promoted.
    An asset whose hint does not make the table fit stays in `review` carrying what the
    model said, so a bad hint is legible rather than silent.
    """
    review = package.get("review") or []
    if not review or client is None:
        return 0

    try:
        payload, _ = client.json_completion(
            system_prompt=REVIEW_SYSTEM_PROMPT, user_prompt=_review_prompt(package))
    except (RuntimeError, ValueError, KeyError) as exc:
        print(f"        review adjudication unavailable ({exc}) — assets stay unresolved")
        return 0

    hints = {}
    for entry in payload.get("assets") or []:
        try:
            hint = AssetHint.model_validate(entry)
        except ValidationError:
            continue
        hints[hint.docling_item_ref] = hint

    promoted, still_open = 0, []
    for asset in review:
        hint = hints.get(asset.get("docling_item_ref"))
        headers = [str(name) for name in asset.get("headers", [])]
        named = {column_label(name) for name in headers}
        if hint is None or hint.not_a_series or not hint.axis_column:
            asset["hint"] = "not a series" if (hint and hint.not_a_series) else "no answer"
            asset["why"] = (hint.why if hint else None) or asset.get("why")
            still_open.append(asset)
            continue
        if hint.axis_column not in named:
            # A header it was never shown means the answer is about another table.
            asset["hint"] = f"named a column that is not there: {hint.axis_column!r}"
            still_open.append(asset)
            continue

        fit = fits_schema(headers, asset.get("rows") or [],
                          prefer_axis=canonical_key(hint.axis_column))
        # Only the keyed fit may be reached this way. A hint naming a column of numbers
        # would otherwise force it onto the axis through the long fit, and the gate had
        # already refused that column for a reason it can state and the model cannot see.
        if not fit.fits or fit.orientation != "keyed":
            asset["hint"] = (f"{hint.axis_column!r} does not key the rows "
                             f"({fit.reason if not fit.fits else fit.orientation + ' fit'})")
            still_open.append(asset)
            continue

        asset["gate"] = fit.summary()
        asset["observations"] = _observation_payload(fit)
        asset["hint"] = f"axis named by the model: {hint.axis_column!r}"
        asset.pop("rows", None)
        (package["figures"] if asset.get("is_figure") else package["tables"]).append(asset)
        promoted += 1

    package["review"] = still_open
    # The report was written before this ran, so every count a promotion moves is redone.
    report = package.get("gate_report", {})
    report["review"] = len(still_open)
    report["review_promoted"] = promoted
    report["tables_accepted"] = len(package.get("tables", []))
    report["figures_accepted"] = len(package.get("figures", []))
    report["observations"] = sum(
        len(item.get("observations", []))
        for item in chain(package.get("tables", []), package.get("figures", [])))
    print(f"        review: {promoted} of {promoted + len(still_open)} assets keyed by the model")
    return promoted


def show_gate_decisions(silver_package: dict) -> pd.DataFrame:
    """Every asset and why it is in or out — the table to read when benchmarking."""
    def row(item, verdict, why, gate=None):
        gate = gate or {}
        return {
            "kind": item.get("kind") or ("table" if "cleaned_csv_path" in item else "figure"),
            "index": item.get("index") or item.get("table_index") or item.get("figure_index"),
            "page": item.get("page_number"),
            "caption": (item.get("caption") or "")[:60],
            "verdict": verdict,
            "why": why,
            "axis": gate.get("axis_label"),
            "points": len(gate.get("axis_points") or []),
            "observations": len(item.get("observations", [])),
        }

    rows = [row(item, "accepted", item.get("gate", {}).get("reason"), item.get("gate"))
            for item in chain(silver_package.get("tables", []), silver_package.get("figures", []))]
    rows += [row(item, "reference", item.get("why"))
             for item in silver_package.get("references", [])]
    rows += [row(item, f"review: {item.get('hint') or 'unresolved'}", item.get("why"))
             for item in silver_package.get("review", [])]
    rows += [row(item, f"rejected @ {item['stage']}", item["reason"])
             for item in silver_package.get("rejected", [])]
    frame = pd.DataFrame(rows).sort_values(["kind", "index"], na_position="last")
    display(frame)
    return frame

## Vocabularies

Data, not code. Four closed sets — everything reaching the database is one of these values or is flagged for review.

In [5]:
# Controlled vocabularies. Data only — nothing below hardcodes a domain term in logic.

INDICATOR_TYPES = ("microbial", "chemical")

FUNCTIONAL_CLASS_TIERS = {
    "carbohydrate": "Macronutrient", "protein": "Macronutrient", "fiber": "Macronutrient",
    "mineral": "Micronutrient",
    "phenol": "Bioactive", "essential oil": "Bioactive", "organic acid": "Bioactive",
}
FUNCTIONAL_CLASSES = tuple(FUNCTIONAL_CLASS_TIERS)
INGREDIENT_SOURCES = ("animal", "plant", "microbial", "mineral", "other")
UNCLASSIFIED_CLASS, UNKNOWN_SOURCE = "unclassified", "unknown"

# `experiments.treatment` is a protocol summary in the paper's own words, free text, so
# no vocabulary constrains it: substances stay in `ingredients` and amounts in
# `experiment_ingredients.concentration`. NULL means "reported as untreated" when an
# `evidence` span names the field, and "never described" when none does.

# How a value was arrived at, and what kind of item supports it. Every `evidence` row
# carries both, so a derived value is never read as one the paper stated outright.
EVIDENCE_METHODS = ("stated", "derived", "inferred")
EVIDENCE_SOURCE_TYPES = ("prose", "table", "figure")

UNRESOLVED_INDICATOR, UNSPECIFIED_UNIT = "unresolved indicator", "unspecified"

# Panel-scored attributes and physical measures — neither a colony count nor a chemical
# assay. Matched before insert, so they reach neither `indicators` nor `measurements`.
DISCARDED_INDICATORS = [
    "sensory score", "sensory evaluation", "sensory analysis", "overall acceptability",
    "overall acceptance", "acceptability", "juiciness", "tenderness", "flavour", "flavor",
    "aroma", "taste", "odour", "odor", "appearance", "texture", "hardness", "chewiness",
    "springiness", "cohesiveness", "gumminess", "colour", "color", "delta e", "de",
    "total colour difference", "total color difference", "l value", "a value", "b value",
    "whiteness", "redness", "yellowness", "lightness",
    "cooking loss", "cooking loss percentage", "weight loss", "drip loss", "purge loss",
    "water holding capacity", "whc",
]

# Two spellings of one unit split one indicator into two rows.
UNIT_ALIASES = {
    "log CFU/g": ["log cfu/g", "log cfu g", "log10 cfu/g", "log cfu/ml", "log cfu/cm2"],
    "mg N/100 g": ["mg n/100g", "mg n/100 g", "mg/100g", "mgn/100g", "mg n per 100 g"],
    "mg MDA/kg": ["mg mda/kg", "mg mda per kg", "mda/kg", "mg malondialdehyde/kg"],
    "meq O2/kg": ["meq o2/kg", "meq/kg", "meq o2/kg fat", "meq o2/kg oil"],
    "CFU/plate": ["cfu/plate", "cfu per plate"],
    "%": ["percent", "% (w/w)", "w/w %", "percentage"],
    "pH": ["ph", "ph units", "ph value"],
    "score": ["sensory score", "points"],
    "ppm": ["mg/l", "µg/ml", "ug/ml"],
}


def matrix(name, *aliases):
    return {"name": name, "kind": "matrix", "aliases": list(aliases)}


def indicator(name, indicator_type, unit, threshold, *aliases):
    return {"name": name, "kind": "indicator", "indicator_type": indicator_type,
            "unit": unit, "threshold": threshold, "aliases": list(aliases)}


def ingredient(name, functional_class, source, *aliases):
    return {"name": name, "kind": "ingredient", "functional_class": functional_class,
            "source": source, "aliases": list(aliases)}


def treatment(name, *aliases):
    return {"name": name, "kind": "treatment", "aliases": list(aliases)}


EO = ("essential oil", "plant")      # every GC-MS constituent shares these

SEED_TERMS = [
    matrix("Chicken fillet", "chicken fillets", "chicken breast fillet"),
    matrix("Ground beef", "minced beef", "beef mince"),
    matrix("Beef", "beef meat"),
    matrix("Chicken", "chicken meat", "poultry meat"),
    matrix("Pork", "pork meat", "minced pork"),
    matrix("Lamb", "lamb meat", "mutton"),
    matrix("Fish fillet", "fish fillets", "carp fillets"),
    matrix("Shrimp", "prawn", "shrimps", "crevette"),

    # Microbial: a count of organisms, always reported in CFU.
    indicator("Total viable count", "microbial", "log CFU/g", 7.0,
              "tvc", "tpc", "tac", "apc", "total viable counts", "total plate count",
              "total bacterial count", "total bacteria count", "aerobic plate count",
              "total aerobic count", "aerobic mesophilic count", "mesophilic count"),
    indicator("Psychrotrophic bacteria count", "microbial", "log CFU/g", 7.0,
              "pbc", "psy", "psychrotrophic count", "psychrotrophs"),
    indicator("Lactic acid bacteria count", "microbial", "log CFU/g", None,
              "lab", "labc", "lactic acid bacteria"),
    indicator("Enterobacteriaceae count", "microbial", "log CFU/g", None,
              "enterobacteriaceae", "enterobacteria"),
    indicator("Total coliform count", "microbial", "log CFU/g", None,
              "tcc", "coliform count", "total coliforms", "coliforms"),
    indicator("Escherichia coli count", "microbial", "log CFU/g", None,
              "e coli", "e coli counts", "escherichia coli", "e coli o157 h7",
              "e coli o157 h7 counts", "escherichia coli o157 h7 counts"),
    indicator("Salmonella count", "microbial", "log CFU/g", None,
              "salmonella", "salmonella counts", "salmonella typhimurium counts",
              "salmonella enterica serovar typhimurium counts"),
    indicator("Staphylococcus aureus count", "microbial", "log CFU/g", None,
              "s aureus", "s aureus counts", "staphylococcus aureus counts"),
    indicator("Pseudomonas count", "microbial", "log CFU/g", None,
              "pseudomonas", "pseudomonas spp", "pseudomonas counts"),
    indicator("Brochothrix thermosphacta count", "microbial", "log CFU/g", None,
              "b thermosphacta", "brochothrix thermosphacta", "brochothrix"),
    indicator("Yeast and mould count", "microbial", "log CFU/g", None,
              "fungal count", "yeast and mold count", "yeasts and moulds",
              "proteolytic fungal count", "lipolytic fungal count", "mould count"),

    # Chemical: everything that is not a count of organisms.
    indicator("Total volatile basic nitrogen", "chemical", "mg N/100 g", 25.0,
              "tvb n", "tvbn", "tvb", "total volatile base nitrogen"),
    indicator("Trimethylamine nitrogen", "chemical", "mg N/100 g", 15.0,
              "tma n", "tman", "tma", "trimethylamine"),
    indicator("Peroxide value", "chemical", "meq O2/kg", 10.0, "pv", "peroxide values"),
    indicator("Thiobarbituric acid reactive substances", "chemical", "mg MDA/kg", 2.0,
              "tbars", "tba", "tbars value", "thiobarbituric acid"),
    indicator("pH", "chemical", "pH", None, "ph value", "ph values"),
    indicator("Water activity", "chemical", "aw", None, "aw", "a w"),
    indicator("Hexanal content", "chemical", "ppm", None, "hc", "hexanal"),
    indicator("Conjugated diene value", "chemical", "absorbance", None,
              "e232", "spectrophotometric indicies e232", "conjugated dienes"),
    indicator("Conjugated triene value", "chemical", "absorbance", None,
              "e270", "spectrophotometric indicies e270", "conjugated trienes"),
    # No sensory, colour or gravimetric terms on purpose — see DISCARDED_INDICATORS.

    # Essential oils and their constituents.
    ingredient("Thyme essential oil", *EO, "teo", "teo1", "teo2", "te0", "thyme oil",
               "thyme eo", "thymus vulgaris oil", "thyme leaves extract", "thyme"),
    ingredient("Sage essential oil", *EO, "seo", "seo1", "seo2", "se0", "seol",
               "sage oil", "sage", "salvia officinalis oil"),
    ingredient("Oregano essential oil", *EO, "oregano eo", "organo eo", "oregano oil",
               "oregano extract", "origanum vulgare oil"),
    ingredient("Clove essential oil", *EO, "clove eo", "clove oil", "clove extract"),
    ingredient("Zataria multiflora essential oil", *EO, "zataria multiflora oil",
               "zataria eo", "shirazi thyme oil"),
    ingredient("Turmeric essential oil", *EO, "turmeric oil", "curcuma oil"),
    ingredient("Rosemary essential oil", *EO, "rosemary oil", "oleroresin rosemary"),
    ingredient("Lemon essential oil", *EO, "lemon oil"),
    ingredient("Lime essential oil", *EO, "lime oil"),
    ingredient("Orange essential oil", *EO, "orange oil"),
    ingredient("Carvacrol", *EO, "carvacrol eo"),
    ingredient("Thymol", *EO, "thymol eo"),
    ingredient("p-Cymene", *EO, "p cymene", "pcymene"),
    # Listed rather than pattern matched: "-ene" is a suffix, not a functional class.
    ingredient("α-Thujene", *EO, "a thujene"),
    ingredient("α-Pinene", *EO, "a pinene"),
    ingredient("β-Pinene", *EO, "b pinene"),
    ingredient("Camphene", *EO),
    ingredient("Camphor", *EO),
    ingredient("Sabinene", *EO),
    ingredient("Myrcene", *EO, "myrecene", "b myrcene"),
    ingredient("α-Terpinene", *EO, "a terpinene"),
    ingredient("γ-Terpinene", *EO, "g terpinene"),
    ingredient("α-Phellandrene", *EO, "a phellandrene"),
    ingredient("β-Phellandrene", *EO, "b phellandrene"),
    ingredient("Limonene", *EO, "limonen", "d limonene"),
    ingredient("Terpinolene", *EO),
    ingredient("Linalool", *EO),
    ingredient("Terpinen-4-ol", *EO, "terpinene 4 ol"),
    ingredient("α-Terpineol", *EO, "a terpineol"),
    ingredient("1,8-Cineole", *EO, "eucalyptol"),
    ingredient("3-Octanone", *EO),
    ingredient("Thymol methyl ether", *EO),
    ingredient("Carvacrol methyl ether", *EO),
    ingredient("Thymol acetate", *EO),
    ingredient("Carvacrol acetate", *EO),
    ingredient("β-Caryophyllene", *EO, "b caryophyllene", "e caryophyllene", "caryophyllene"),
    ingredient("β-Bisabolene", *EO, "b bisabolene"),
    ingredient("Spathulenol", *EO),
    ingredient("Viridiflorene", *EO),
    ingredient("Aromadendrene", *EO),
    ingredient("Eugenol", *EO),
    ingredient("Cinnamaldehyde", *EO, "trans cinnamaldehyde"),
    ingredient("Menthol", *EO),

    # Phenolic extracts.
    ingredient("Moringa oleifera leaves extract", "phenol", "plant", "moringa extract",
               "moringa oleifera extract", "moringa", "drumstick leaf",
               "dry drumstick flower moringa oleifera l"),
    ingredient("Rosemary extract", "phenol", "plant", "rosemary", "roremary",
               "extract of rosemary", "rosemary oil extract", "rosemary water extract",
               "rosemary precooking", "rosemary postcooking"),
    ingredient("Green tea extract", "phenol", "plant", "tea polyphenol",
               "tea polyphenols", "tea catechins", "green tea"),
    ingredient("Grape seed extract", "phenol", "plant", "grape seed", "activin",
               "grape and blueberry"),
    ingredient("Grape pomace extract", "phenol", "plant", "grape pomace extract 1",
               "grape pomace extract 2"),
    ingredient("Pomegranate peel extract", "phenol", "plant", "pomegranate fruit juice",
               "pomegranate extract"),
    ingredient("Cranberry extract", "phenol", "plant", "cranberry", "cranberry pomace extracts"),
    ingredient("Lime extract", "phenol", "plant", "lime"),
    ingredient("Garlic extract", "phenol", "plant", "garlic"),
    ingredient("Olive leaf extract", "phenol", "plant", "olive leaf",
               "phenol extract from olive oil mill wastewater"),
    ingredient("Propolis extract", "phenol", "animal", "ethanolic extract of propolis eep",
               "eep", "propolis"),
    ingredient("Ellagic acid", "phenol", "plant"),
    ingredient("Sesamol", "phenol", "plant"),
    ingredient("Pycnogenol", "phenol", "plant"),
    ingredient("Butylated hydroxytoluene", "phenol", "other", "bht"),
    ingredient("Butylated hydroxyanisole", "phenol", "other", "bha"),
    ingredient("Cacao bean husk powder", "phenol", "plant", "cacao bean husk"),
    ingredient("Honeybush extract", "phenol", "plant", "unfermented honeybush extract"),
    ingredient("Citrus peel extract", "phenol", "plant"),
    ingredient("Potato peel extract", "phenol", "plant", "potato peel ethanol extract",
               "potato peel water extract"),
    ingredient("Bee pollen", "phenol", "animal", "lyophilized bee pollen"),

    # Organic acids.
    ingredient("Lactic acid", "organic acid", "microbial"),
    ingredient("Peracetic acid", "organic acid", "other", "acide peracetique", "pa"),
    ingredient("Fumaric acid", "organic acid", "other"),
    ingredient("EDTA", "organic acid", "other", "na2edta", "disodium edta",
               "ethylenediaminetetraacetic acid"),
    ingredient("Sodium benzoate", "organic acid", "other"),
    ingredient("Sodium ascorbate", "organic acid", "other", "sodium erythorbate", "ascorbate"),
    ingredient("Lemon juice", "organic acid", "plant"),

    # Proteins. A bacteriocin is a peptide, and there is no peptide leaf.
    ingredient("Nisin", "protein", "microbial", "nisine", "n5", "n10"),
    ingredient("Brevibacillin", "protein", "microbial", "brevibacilline", "b5", "b10"),
    ingredient("Lysozyme", "protein", "animal", "lysozyme film"),
    ingredient("Lactoferrin", "protein", "animal"),
    ingredient("Whey protein concentrate", "protein", "animal", "whey protein"),
    ingredient("Sodium caseinate", "protein", "animal", "caseinate"),
    ingredient("Soy protein", "protein", "plant", "soya protein"),

    # Carbohydrates, fibre, minerals.
    ingredient("Chitosan", "carbohydrate", "animal", "chitosane"),
    ingredient("Inulin", "fiber", "plant"),
    ingredient("Pea fibre", "fiber", "plant", "pea fiber"),
    ingredient("Lemon fibre", "fiber", "plant", "lemon fiber"),
    ingredient("Orange fibre", "fiber", "plant", "orange fiber"),
    ingredient("Sodium chloride", "mineral", "mineral", "nacl", "salt",
               "anodic nacl solution", "cathodic nacl solution"),
    ingredient("Sodium tripolyphosphate", "mineral", "mineral", "stpp"),
    ingredient("Sodium nitrate", "mineral", "mineral", "nitrate"),
    ingredient("Sulphur dioxide", "mineral", "mineral", "so2", "sulfur dioxide"),
    ingredient("Chlorine dioxide", "mineral", "mineral", "clo2"),

    # Interventions, matched inside a label, so "MAP-packed" needs no alias. These
    # resolve ARM LABELS in Silver; `experiments.treatment` holds protocol prose instead.
    treatment("Refrigerated storage", "refrigerated", "chilled storage", "cold storage"),
    treatment("Frozen storage", "frozen", "freezing", "deep frozen"),
    treatment("Heat treatment", "heat", "heated", "thermal treatment", "cooking", "cooked",
              "pasteurisation", "pasteurization", "sous vide", "blanching", "precooking"),
    treatment("Irradiation", "irradiated", "gamma irradiation", "e beam", "electron beam",
              "uv c", "uv irradiation"),
    treatment("Modified atmosphere packaging", "map", "modified atmosphere",
              "modified atmosphere packed", "gas flushing", "co2 packaging"),
    treatment("Vacuum packaging", "vacuum", "vacuum packed", "vacuum sealed"),
    treatment("Active packaging", "active film", "antimicrobial packaging",
              "active antimicrobial", "antimicrobial film"),
    treatment("Edible coating", "coating", "coated", "edible film", "film coating"),
    treatment("High pressure processing", "hpp", "high pressure", "high hydrostatic pressure"),
]

KINDS = ("indicator", "ingredient", "matrix", "treatment")
print(f"Seed vocabulary: {len(SEED_TERMS)} terms across {len(KINDS)} kinds")

Seed vocabulary: 125 terms across 4 kinds


## Silver: Normalisation

The only place names are resolved. Gold receives a package where the matrix, the arms, their concentrations, the indicator names, types and units, and every ingredient's class and origin are already decided.

In [6]:
from collections import Counter

# Silver, part 2: structure becomes meaning. Of the labels on a value, which names the
# INDICATOR and which the TREATMENT? Answered from two structural facts — a label parsing
# as substance + amount + unit heads an arm, and labels beside one are arms too — plus
# the vocabulary. A dosed label names an ingredient, not an intervention: its substances
# go to `ingredients` and its amount to `experiment_ingredients.concentration`.

VOCABULARY_PATH = PROJECT_ROOT / "vocabulary.json"
_CFU_UNIT = re.compile(r"\bcfu\b", re.IGNORECASE)
_DISCARDED = re.compile(
    "(?:^|_)(" + "|".join(sorted((canonical_key(name) for name in DISCARDED_INDICATORS),
                                 key=len, reverse=True)) + ")(?:_|$)")
_UNIT_CANONICAL = {canonical_key(alias): preferred
                   for preferred, aliases in UNIT_ALIASES.items()
                   for alias in [preferred, *aliases]}
_KNOWN_UNIT_KEYS = {canonical_key(preferred) for preferred in UNIT_ALIASES}


def canonical_unit(unit):
    text = str(unit or "").strip()
    return _UNIT_CANONICAL.get(canonical_key(text), text) if text else None


@dataclass(eq=False)
class Term:
    key: str
    name: str
    kind: str
    unit: Optional[str] = None
    threshold: Optional[float] = None
    functional_class: Optional[str] = None
    source: Optional[str] = None
    indicator_type: Optional[str] = None
    aliases: list = field(default_factory=list)

    def to_json(self) -> dict:
        payload = {"name": self.name, "kind": self.kind, "aliases": sorted(set(self.aliases))}
        for attribute in ("unit", "threshold", "functional_class", "source", "indicator_type"):
            if getattr(self, attribute) is not None:
                payload[attribute] = getattr(self, attribute)
        return payload


class Vocabulary:
    """Alias -> preferred term, persisted as JSON, and the ledger of what it changed."""

    def __init__(self):
        self._by_kind = {kind: {} for kind in KINDS}
        self.unresolved = {kind: {} for kind in KINDS}
        self.changes, self.flags = {}, {}
        self.discarded = Counter()
        self._pattern = {}

    def add(self, term: Term) -> None:
        table = self._by_kind.setdefault(term.kind, {})
        for key in [term.key, *(canonical_key(alias) for alias in term.aliases)]:
            taken = table.get(key)
            if taken is not None and taken.name != term.name:
                # Two terms on one key means the second answers every lookup for the
                # first, silently. That is how α- and γ-terpinene merged.
                self.flag(term.kind, term.name, f"key {key!r} already claimed by {taken.name!r}")
            table[key] = term
        self._pattern.pop(term.kind, None)

    def _compile(self, kind):
        """One alternation per kind, not one regex per alias per lookup — `find_in` runs
        thousands of times and blew through Python's 512-entry regex cache. Longest
        alternatives first, so "aerobic plate count" beats "count"."""
        keys = sorted((key for key in self._by_kind.get(kind, {}) if len(key) >= 2),
                      key=len, reverse=True)
        self._pattern[kind] = re.compile(
            "(?:^|_)(" + "|".join(re.escape(key) for key in keys) + ")(?:_|$)"
        ) if keys else None
        return self._pattern[kind]

    def pattern(self, kind):
        return self._pattern[kind] if kind in self._pattern else self._compile(kind)

    @classmethod
    def load(cls, path: Path, seed=None) -> "Vocabulary":
        """Seeded on first use and never after, so edits to the file survive."""
        if path.exists():
            payload = json.loads(path.read_text(encoding="utf-8"))
        else:
            payload = {"terms": list(seed or ()), "review": {}}
            path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
        vocabulary = cls()
        for entry in payload.get("terms", []):
            vocabulary.add(Term(
                key=canonical_key(entry.get("key") or entry["name"]),
                name=entry["name"], kind=entry["kind"], unit=entry.get("unit"),
                threshold=entry.get("threshold"),
                functional_class=entry.get("functional_class"), source=entry.get("source"),
                indicator_type=entry.get("indicator_type"), aliases=entry.get("aliases", []),
            ))
        return vocabulary

    def save(self, path: Path) -> None:
        seen, terms = set(), []
        for table in self._by_kind.values():
            for term in table.values():
                if id(term) not in seen:
                    seen.add(id(term))
                    terms.append(term.to_json())
        path.write_text(json.dumps({
            "terms": sorted(terms, key=lambda entry: (entry["kind"], entry["name"])),
            "review": {kind: sorted(names) for kind, names in self.unresolved.items() if names},
        }, ensure_ascii=False, indent=2), encoding="utf-8")

    def resolve(self, text, kind) -> Optional[Term]:
        return self._by_kind.get(kind, {}).get(canonical_key(text))

    def names_in(self, text, kind) -> set:
        """Every known term of `kind` appearing in a phrase, longest match per span."""
        key = canonical_key(text)
        pattern = self.pattern(kind) if key else None
        if pattern is None:
            return set()
        table = self._by_kind[kind]
        return {table[match.group(1)] for match in pattern.finditer(key)}

    def find_in(self, text, kind) -> Optional[Term]:
        """Longest known term appearing anywhere in a phrase, for captions and prose."""
        key = canonical_key(text)
        pattern = self.pattern(kind) if key else None
        if pattern is None:
            return None
        best = max((match.group(1) for match in pattern.finditer(key)), key=len, default=None)
        return self._by_kind[kind][best] if best else None

    def terms_of(self, kind) -> dict:
        return self._by_kind.get(kind, {})

    def note_unresolved(self, text, kind) -> None:
        text = str(text or "").strip()
        if text:
            self.unresolved.setdefault(kind, {})[canonical_key(text)] = text

    def note_change(self, kind, raw, canonical, **extra) -> None:
        raw = str(raw or "").strip()
        if raw:
            self.changes[(kind, canonical_key(raw))] = {
                "kind": kind, "raw": raw, "canonical": canonical, **extra}

    def flag(self, kind, value, reason) -> None:
        value = str(value or "").strip()
        if value:
            self.flags[(kind, value)] = reason

    def normalise_ingredient(self, raw) -> Optional[dict]:
        """name + functional_class + source together: one decision, not three. Returns
        None for a substance the vocabulary does not know — a sensory table has the shape
        of a catalogue too — and parks the name under `review` for promotion."""
        text = str(raw or "").strip()
        term = self.resolve(text, "ingredient")
        if term is None:
            self.note_unresolved(text, "ingredient")
            self.discarded[("ingredient", text)] += 1
            return None
        klass = term.functional_class if term.functional_class in FUNCTIONAL_CLASSES else None
        source = term.source if term.source in INGREDIENT_SOURCES else None
        for field_name, value in (("functional_class", klass), ("source", source)):
            if value is None:
                self.flag("ingredient", term.name, f"{field_name} outside its vocabulary")
        resolved = {"name": term.name, "functional_class": klass or UNCLASSIFIED_CLASS,
                    "source": source or UNKNOWN_SOURCE}
        self.note_change("ingredient", text, term.name, **{
            k: v for k, v in resolved.items() if k != "name"})
        return resolved

    def normalise_indicator(self, raw_label) -> Optional[dict]:
        """The unit is split off before lookup, then the vocabulary's unit wins. None
        means a discarded quantity and the caller drops the observation. An *unknown*
        indicator is kept, unlike an unknown ingredient: a measurement nobody has named is
        still a measurement, so it keeps the paper's wording and is flagged."""
        raw_name, raw_unit = split_label_and_unit(raw_label)
        if not raw_name:
            return {"name": UNRESOLVED_INDICATOR, "indicator_type": "chemical",
                    "unit": UNSPECIFIED_UNIT, "threshold": None}
        if _DISCARDED.search(canonical_key(raw_name)):
            self.discarded[("indicator", raw_name)] += 1
            return None
        term = self.resolve(raw_name, "indicator")
        unit = canonical_unit(term.unit if term and term.unit else raw_unit) or UNSPECIFIED_UNIT
        # A colony count is reported in CFU and nothing else is, so the unit is the one
        # structural signal that classifies a term the vocabulary misses.
        inferred = "microbial" if _CFU_UNIT.search(unit) else "chemical"
        if term is None:
            self.note_unresolved(raw_name, "indicator")
            self.flag("indicator", raw_name, f"not in vocabulary; type inferred {inferred!r}")
            return {"name": raw_name, "indicator_type": inferred, "unit": unit, "threshold": None}
        indicator_type = term.indicator_type
        if indicator_type not in INDICATOR_TYPES:
            indicator_type = inferred
            self.flag("indicator", term.name, f"no valid indicator_type; inferred {inferred!r}")
        self.note_change("indicator", raw_label, term.name,
                         indicator_type=indicator_type, unit=unit)
        return {"name": term.name, "indicator_type": indicator_type, "unit": unit,
                "threshold": term.threshold}

    def normalise_treatment(self, text) -> Optional[str]:
        term = self.find_in(text, "treatment")
        if term:
            self.note_change("treatment", text, term.name)
        return term.name if term else None


# ─── Arms ─────────────────────────────────────────────────────────────────────

_LABEL_SEPARATORS = (" - ", " – ", " — ", ": ", " | ", " _ ")
# Chart OCR reads o/l/s as 0/1/5. Inside a substance name — the dose already split off —
# a digit touching letters can only be one of those misreads.
_OCR_DIGITS = {"0": "o", "1": "l", "5": "s"}
_OCR_IN_WORD = re.compile(r"(?<=[A-Za-z])[015](?=[A-Za-z]|$|[+/&\s])")


def unfold_ocr_digits(text):
    return _OCR_IN_WORD.sub(lambda match: _OCR_DIGITS[match.group()], text)


@dataclass
class Treatment:
    """One arm. `ingredients` is the additive dimension — empty means a control — and
    `treatment` is the process this arm's own label names."""

    key: str
    label: str                      # as printed
    treatment: Optional[str]        # the process the LABEL names, not the protocol
    ingredients: list = field(default_factory=list)

    @property
    def dosed(self):
        return bool(self.ingredients)


def _dosed_ingredients(dose: Dose, vocabulary):
    """The substances a dosed label names, each with its own concentration."""
    if not (dose and dose.substance):
        return []
    found = []
    # '+' and '&' combine substances; '/' does not — it is the slash in "log cfu/g".
    for part in re.split(r"\s*[+&]\s*", unfold_ocr_digits(dose.substance)):
        part = part.strip(" -_,")
        resolved = vocabulary.normalise_ingredient(part) if part and vocabulary else None
        if resolved:
            found.append({**resolved, "amount": dose.amount, "unit": dose.unit})
    return found


def _dose_key(ingredients):
    """What separates one rung of a dose ladder from the next."""
    return "|".join(f"{item['name']}={item['amount']}{item['unit']}"
                    for item in sorted(ingredients, key=lambda item: item["name"]))


def parse_treatment(label, vocabulary=None) -> Treatment:
    """Read one arm: its substances, their concentrations, and its condition. The dose
    never enters a name — it rides on each ingredient so it lands in `concentration` —
    and the arm's identity is its substances *and* their amounts."""
    text = _DUPLICATE_TAIL.sub("", str(label or "").strip())
    dose = parse_dosed_label(text)
    ingredients = _dosed_ingredients(dose, vocabulary)
    # Decided from THIS arm's label alone. A methods sentence "samples were vacuum
    # packed" covers every arm including the control, so prose is reported by
    # `resolve_paper_terms` and never written.
    named = vocabulary.normalise_treatment(text) if vocabulary else None
    if vocabulary is not None and dose and not ingredients:
        # The label carries a dose but none of its substances resolved. Without this the
        # arm has no ingredients and is indistinguishable from a control.
        vocabulary.flag("treatment", text,
                        "dosed arm whose substance is not in the vocabulary — "
                        "reads as a control until the term is added")
    substances = " + ".join(item["name"] for item in ingredients)
    return Treatment(
        key=canonical_key(f"{named or ''}|{substances}|{_dose_key(ingredients)}")
            or canonical_key(text) or "unspecified",
        label=text, treatment=named, ingredients=ingredients)


def _has_separator(label):
    return any(separator in label for separator in _LABEL_SEPARATORS)


def split_indicator_and_treatment(label, lexicon):
    """Split "PBC (log cfu/g) - Control1" when the tail is a known arm."""
    text = str(label or "").strip()
    for separator in _LABEL_SEPARATORS:
        if separator in text:
            head, _, tail = text.rpartition(separator)
            if canonical_key(tail) in lexicon:
                return head.strip(), tail.strip()
    return text, None


def _candidate_columns(asset):
    """The labels on this asset that could name an arm, as separate groups.

    Two places an arm can be printed: the value column headers, and the row labels — a
    keyed table puts them there, since "control, 6 days" leaves "control" behind, so
    reading only the headers finds no arm at all on exactly the tables the keyed fit
    exists to recover.

    They stay apart because `_collect_arms` reads a dose anywhere in a group as evidence
    that the group's other labels are arms too. Pooled, one dosed row label would make
    every indicator column beside it an arm.
    """
    gate = asset.get("gate") or {}
    axis = {canonical_key(part) for part in str(gate.get("axis_label") or "").split("|")}

    def keep(names):
        # The axis is never an arm; one misread time column puts "Storage time" in the
        # lexicon and from there into `experiments`.
        return [name for name in names if canonical_key(name) not in axis]

    headers = keep(str(name) for name in gate.get("value_columns", []))
    labels, seen = [], set()
    for observation in asset.get("observations", []):
        for value in (observation.get("row_labels") or {}).values():
            if value not in seen:
                seen.add(value)
                labels.append(str(value))
    return [headers, keep(labels)]


def _collect_arms(columns_per_asset, vocabulary, parsed):
    """An arm carries a dose, or stands beside one. Recurrence alone is not enough: a
    paper measuring pH in two tables makes "pH" recur exactly like an arm. `parsed`
    caches by canonical key across both passes — headers repeat across a paper."""
    arms, dosed, siblings = {}, set(), set()
    for columns in columns_per_asset:
        keyed = []
        for name in columns:
            key = canonical_key(name)
            if key not in parsed:
                parsed[key] = parse_treatment(name, vocabulary)
            keyed.append((key, parsed[key]))
        has_dose = any(arm.dosed for _, arm in keyed)
        for key, arm in keyed:
            arms.setdefault(key, arm)
            if arm.dosed:
                dosed.add(key)
            elif has_dose:
                siblings.add(key)   # undosed but beside a dose ladder: the control
    return {key: arm for key, arm in arms.items() if key in dosed or key in siblings}


def build_treatment_lexicon(assets, vocabulary=None) -> dict:
    """The paper's arms, keyed by the label as printed. Two passes: plain headers first,
    then "PBC (log cfu/g) - Teo1%" unpacked using them — otherwise that reads as an arm
    per indicator, with the unit inside a substance name."""
    columns = [group for asset in assets for group in _candidate_columns(asset)]
    parsed: dict = {}
    lexicon = _collect_arms(
        [[name for name in group if not _has_separator(name)] for group in columns],
        vocabulary, parsed)
    lexicon.update(_collect_arms([
        [split_indicator_and_treatment(name, lexicon)[1] or name
         for name in group if _has_separator(name)]
        for group in columns], vocabulary, parsed))
    return lexicon


# ─── Matrix, interventions, indicators ────────────────────────────────────────

_CAPTION_PREFIX = re.compile(r"^\s*(?:fig(?:ure)?|table|scheme|chart)\s*\.?\s*\d+\s*[.:)-]*\s*",
                             re.IGNORECASE)


def _paper_texts(package):
    """Title, methods and captions, in document order."""
    for section in package.get("sections", []):
        yield f"section:{section['section_title']}"[:80], section["section_title"]
        yield f"section:{section['section_title']}"[:80], section["content_markdown"][:2000]
    for asset in chain(package.get("tables", []), package.get("figures", [])):
        if asset.get("caption"):
            yield asset.get("docling_item_ref") or "caption", asset["caption"]


def resolve_paper_terms(package, vocabulary) -> tuple:
    """The matrix and the interventions the prose names, in one pass. Interventions are
    reported, never assigned: a paper mentioning irradiation does not mean *this* arm
    was irradiated."""
    matrix = {"name": None, "evidence": None}
    interventions = {}
    for where, text in _paper_texts(package):
        if matrix["name"] is None:
            term = vocabulary.find_in(text, "matrix")
            if term:
                matrix = {"name": term.name, "evidence": where}
        term = vocabulary.find_in(text, "treatment")
        if term and term.name not in interventions:
            interventions[term.name] = where
    return matrix, [{"name": name, "evidence": where}
                    for name, where in interventions.items()]


def caption_indicator(caption, vocabulary) -> Optional[str]:
    term = vocabulary.find_in(_CAPTION_PREFIX.sub("", str(caption or "").strip()), "indicator")
    return term.name if term else None


def caption_is_discarded(caption, vocabulary) -> bool:
    """The whole asset measures something we do not keep.

    "Sensory evaluation of chicken fillets" names one quantity and it is discarded, so
    every observation under it would otherwise fall through to `unresolved indicator`.
    But a caption that lists what a table holds — "Changes in Hardness, pH, HCl Titrate,
    DM, Aw, and APC" — names a discarded quantity *and* several kept ones, and dropping
    the table on the first would throw away five columns to avoid one. So the discard
    only stands when nothing in the caption is a known indicator; a single discarded
    column is dropped per observation by `normalise_indicator`, where it belongs.
    """
    text = _CAPTION_PREFIX.sub("", str(caption or "").strip())
    if not _DISCARDED.search(canonical_key(text)):
        return False
    return caption_indicator(text, vocabulary) is None


def context_indicator(context, vocabulary) -> Optional[str]:
    """For a figure with no caption, and only when the surrounding text names exactly one
    known quantity: picking one of several would misattribute real numbers."""
    found = {term.name for term in vocabulary.names_in(context, "indicator")}
    return found.pop() if len(found) == 1 else None


@dataclass(slots=True)
class Measurement:
    axis_value: float
    treatment: Optional[str]     # the arm label's process, if it named one
    arm_key: str
    indicator: str
    indicator_type: str
    unit: str
    threshold: Optional[float]
    value: float
    is_figure: bool          # provenance for the evidence row, not a measurement attribute
    item_ref: Optional[str]
    page_number: Optional[int]
    ingredients: list = field(default_factory=list)


def _assign_roles(observation, lexicon, indicator_columns, strategies):
    """Which label names the quantity and which names the arm. The whole inversion."""
    column = observation.get("column_label")
    indicator_label = treatment_label = None
    if column:
        head, tail = split_indicator_and_treatment(column, lexicon)
        if tail is not None:
            indicator_label, treatment_label = head, tail
            strategies.add("indicator and arm packed into the column header")
        elif canonical_key(column) in lexicon:
            treatment_label = column
            strategies.add("value columns are arms")
        else:
            indicator_label = column
            strategies.add("value columns are indicators")
    for name, value in (observation.get("row_labels") or {}).items():
        if canonical_key(value) in lexicon or canonical_key(name) in lexicon:
            treatment_label = treatment_label or value
            strategies.add("row labels are arms")
        elif name in indicator_columns and indicator_label is None:
            indicator_label = value
            strategies.add("row labels name the indicator")
        elif treatment_label is None:
            # The column named the quantity, so the row qualifies the arm: a study whose
            # arms are named rather than dosed ("Control", "Coated").
            treatment_label = value
            strategies.add("row labels qualify the arm")
    return indicator_label, treatment_label


def interpret_asset(asset, lexicon, vocabulary, *, caption_name=None, claimed=()):
    gate = asset.get("gate") or {}
    observations = asset.get("observations", [])
    if caption_is_discarded(asset.get("caption"), vocabulary):
        name = _CAPTION_PREFIX.sub("", str(asset.get("caption")).strip())[:60]
        vocabulary.discarded[("indicator", name)] = len(observations)
        return [], {"item_ref": asset.get("docling_item_ref"), "caption": name,
                    "how": ["discarded: caption names a sensory or gravimetric quantity"],
                    "indicators": [], "treatments": [], "arms": 0}
    from_context = False
    if caption_name is None:
        guess = context_indicator(asset.get("context_markdown"), vocabulary)
        if guess and canonical_key(guess) not in claimed:
            caption_name, from_context = guess, True

    labels = [str(name) for name in gate.get("label_columns", [])]
    indicator_columns = [name for name in labels if canonical_key(name) not in lexicon]
    is_figure = bool(asset.get("is_figure"))
    item_ref = asset.get("docling_item_ref")
    page_number = asset.get("page_number")
    measurements, strategies, dropped = [], set(), 0

    for observation in observations:
        indicator_label, treatment_label = _assign_roles(
            observation, lexicon, indicator_columns, strategies)
        if indicator_label is None and caption_name:
            indicator_label = caption_name
            strategies.add("indicator inferred from nearby text" if from_context
                           else "indicator taken from the caption")
        resolved = vocabulary.normalise_indicator(indicator_label)
        if resolved is None:
            dropped += 1          # sensory or gravimetric: not a shelf-life indicator
            continue
        arm = lexicon.get(canonical_key(treatment_label)) if treatment_label else None
        measurements.append(Measurement(
            axis_value=float(observation["axis_value"]),
            treatment=arm.treatment if arm else None,
            arm_key=arm.key if arm else canonical_key(treatment_label or "unspecified"),
            indicator=resolved["name"], indicator_type=resolved["indicator_type"],
            unit=resolved["unit"], threshold=resolved["threshold"],
            value=float(observation["value"]), is_figure=is_figure,
            item_ref=item_ref, page_number=page_number,
            ingredients=arm.ingredients if arm else [],
        ))

    if dropped:
        strategies.add(f"{dropped} observations of discarded indicators dropped")
    return measurements, {
        "item_ref": item_ref, "caption": (asset.get("caption") or "")[:80],
        "how": sorted(strategies) or ["no labels to resolve"],
        "indicators": sorted({m.indicator for m in measurements}),
        # Only the arms whose label named a process: `treatment` is None for the rest,
        # and an asset holding both kinds cannot be sorted.
        "treatments": sorted({m.treatment for m in measurements if m.treatment}),
        "arms": len({m.arm_key for m in measurements}),
    }


def interpret_paper(assets, lexicon, vocabulary):
    """Captions are read across the paper first: an indicator another figure claimed by
    caption is evidence against a captionless one guessing the same name from prose."""
    caption_names = [caption_indicator(asset.get("caption"), vocabulary) for asset in assets]
    claimed = {canonical_key(name) for name in caption_names if name}
    measurements, notes = [], []
    for asset, caption_name in zip(assets, caption_names):
        found, note = interpret_asset(asset, lexicon, vocabulary,
                                      caption_name=caption_name, claimed=claimed)
        measurements.extend(found)
        notes.append(note)
    return measurements, notes


# ─── Composition tables ───────────────────────────────────────────────────────

_PROPORTION_UNITS = ("%", "peak area %", "area %", "w/w", "v/v", "g/100g")


def _column_is_numeric(rows, index):
    filled = [row[index] for row in rows
              if index < len(row) and row[index] is not None and str(row[index]).strip()]
    return bool(filled) and sum(parse_number(cell) is not None for cell in filled) / len(filled) >= 0.6


def _proportion_unit(header) -> Optional[str]:
    """A unit, or nothing. The bracket at the end of a header is not always one:
    "Concentration (mean ± Stdev)" gave `concentration_unit = "mean ± Stdev"`."""
    _, unit = split_label_and_unit(header)
    candidate = canonical_unit(unit) if unit else None
    if candidate and canonical_key(candidate) in _KNOWN_UNIT_KEYS:
        return candidate
    return next((token for token in _PROPORTION_UNITS
                 if token in str(header or "").lower()), None)


_CATALOGUE_TOTALS = {"total", "sum", "others", "other", "unknown"}


def ingredients_from_reference(asset, vocabulary) -> list:
    """A composition table's rows are substances, so they belong in `ingredients`. Only
    rows the vocabulary can name survive, since a sensory table has the shape of a
    catalogue too. The column group prefix ("TEO.Concentration") names the preparation
    and is kept as provenance; `source` is the origin and comes from the vocabulary."""
    headers = [str(name) for name in asset.get("headers", [])]
    rows = asset.get("rows") or []
    if not headers or not rows:
        return []
    numeric = [_column_is_numeric(rows, index) for index in range(len(headers))]
    name_columns = [index for index, flag in enumerate(numeric) if not flag]
    amount_columns = [index for index, name in enumerate(headers)
                      if numeric[index]
                      and any(token in name.lower() for token in _PROPORTION_UNITS)]
    caption = asset.get("caption")
    item_ref = asset.get("docling_item_ref")
    found = []
    for name_index in name_columns:
        header = headers[name_index]
        prefix = header.split(".")[0].strip() if "." in header else None
        partner = next((index for index in amount_columns
                        if prefix and headers[index].startswith(prefix + ".")),
                       amount_columns[0] if amount_columns else None)
        unit = _proportion_unit(headers[partner]) if partner is not None else None
        preparation = prefix or caption or "composition table"
        for row in rows:
            raw = str(row[name_index] or "").strip() if name_index < len(row) else ""
            if not raw or canonical_key(raw) in _CATALOGUE_TOTALS:
                continue
            resolved = vocabulary.normalise_ingredient(raw)
            if resolved is None:
                continue
            found.append({
                **resolved,
                "preparation": preparation,
                "amount": parse_number(row[partner]) if partner is not None and partner < len(row) else None,
                "unit": unit,
                "item_ref": item_ref,
            })
    return found


# ─── The Silver normalisation stage ───────────────────────────────────────────

def normalise_silver(package: dict) -> dict:
    """Normalise one gated Silver package in place and return what was decided: the
    matrix, the arms and their concentrations, the indicator names, types and units, and
    every ingredient's class and origin. Gold assembles records; it resolves nothing."""
    assets = package.get("tables", []) + package.get("figures", [])
    lexicon = build_treatment_lexicon(assets, VOCABULARY)
    measurements, notes = interpret_paper(assets, lexicon, VOCABULARY)
    matrix, interventions = resolve_paper_terms(package, VOCABULARY)
    reading = {
        "matrix": matrix,
        "interventions": interventions,
        "catalogue": [item for reference in package.get("references", [])
                      for item in ingredients_from_reference(reference, VOCABULARY)],
        "lexicon": lexicon, "measurements": measurements, "notes": notes,
    }
    package["reading"] = reading
    VOCABULARY.save(VOCABULARY_PATH)
    return reading


VOCABULARY = Vocabulary.load(VOCABULARY_PATH, seed=SEED_TERMS)
print(f"Vocabulary: {sum(len(VOCABULARY.terms_of(k)) for k in KINDS)} lookup keys "
      f"from {VOCABULARY_PATH.name}")

Vocabulary: 413 lookup keys from vocabulary.json


## Gold

Schema-first assembly: every record is validated before SQLite sees it, and free-form model output never writes directly. The local model is asked only for the two fields that exist solely in prose.

In [7]:
from typing import Literal

from pydantic import (
    BaseModel, ConfigDict, Field, ValidationError, field_validator, model_validator,
)

class EvidenceSpan(BaseModel):
    """What supports one extracted field, and how far it sits from the paper's words.

    `field_name` makes a span attributable: an experiment carries several. `method` is
    the honest part — `stated`, `derived` from operands the paper supplies, or `inferred`
    on weaker grounds, the last two needing a rationale. `confidence` is not the model's
    to choose: `score_evidence` overwrites it.
    """

    model_config = ConfigDict(extra="forbid")
    field_name: str                 # "treatment" | "weight_g" | "indicator_value" | ...
    docling_item_ref: str
    page_number: int | None = None
    source_type: Literal["prose", "table", "figure"]
    source_label: str | None = None
    exact_text: str | None = None
    method: Literal["stated", "derived", "inferred"]
    rationale: str | None = None
    confidence: float = Field(default=0.0, ge=0.0, le=1.0)
    value_is_approximate: bool = False

    @model_validator(mode="after")
    def _attribution_is_complete(self):
        if self.method != "stated":
            assert (self.rationale or "").strip(), \
                f"{self.method} evidence for {self.field_name!r} needs a rationale: a " \
                "value the paper does not state outright carries the reasoning that got there"
        elif self.source_type == "prose":
            assert (self.exact_text or "").strip(), \
                f"stated prose evidence for {self.field_name!r} needs exact_text: the " \
                "sentence the paper states it in, so the claim can be checked"
        return self

class PaperDocument(BaseModel):
    model_config = ConfigDict(extra="forbid")
    doi: str | None = None
    title: str
    abstract: str | None = None
    published_year: int | None = None

class SectionDocument(BaseModel):
    model_config = ConfigDict(extra="forbid")
    section_title: str
    content_markdown: str
    embedding: str | None = None
    docling_item_ref: str | None = None

class TableDocument(BaseModel):
    model_config = ConfigDict(extra="forbid")
    caption: str | None = None
    csv_filepath: str
    structured_json: dict[str, Any] = Field(default_factory=dict)
    docling_item_ref: str | None = None

class FigureDocument(BaseModel):
    model_config = ConfigDict(extra="forbid")
    caption: str | None = None
    image_filepath: str
    semantic_tags: list[str] = Field(default_factory=list)
    docling_item_ref: str | None = None

class IngredientRecord(BaseModel):
    """These validate rather than coerce: a class outside the seven leaves, or a source
    that is really a table name, is a data problem best seen here."""

    model_config = ConfigDict(extra="forbid")
    ingredient_name: str
    functional_class: str
    source: str

    @field_validator("functional_class")
    @classmethod
    def _class(cls, value):
        assert value in FUNCTIONAL_CLASSES + (UNCLASSIFIED_CLASS,), \
            f"functional_class {value!r} not in {list(FUNCTIONAL_CLASSES)}"
        return value

    @field_validator("source")
    @classmethod
    def _source(cls, value):
        assert value in INGREDIENT_SOURCES + (UNKNOWN_SOURCE,), \
            f"source {value!r} not in {list(INGREDIENT_SOURCES)}; it is an origin, " \
            "not the table or arm the substance was read from"
        return value

class ExperimentIngredientRecord(BaseModel):
    """Where the dose lives. `concentration` is the only place an amount is stored."""

    model_config = ConfigDict(extra="forbid")
    ingredient_name: str
    concentration: float | None = None
    concentration_unit: str | None = None

class IndicatorRecord(BaseModel):
    model_config = ConfigDict(extra="forbid")
    indicator_name: str
    indicator_type: Literal["microbial", "chemical"]
    indicator_unit: str
    indicator_threshold: float | None = None

class MeasurementRecord(BaseModel):
    model_config = ConfigDict(extra="forbid")
    day: int
    indicator_name: str
    indicator_type: Literal["microbial", "chemical"]
    indicator_unit: str
    indicator_value: float
    indicator_threshold: float | None = None

class ExperimentRecord(BaseModel):
    """One arm, in two dimensions: `treatment` is the protocol it went through, in prose,
    and the additive dimension is `ingredients` + `experiment_ingredients`. Both nullable
    fields are None for two reasons only `evidence` tells apart — a span with the matching
    `field_name` means the paper addressed it, none means it never did."""

    model_config = ConfigDict(extra="forbid")
    meat_matrix: str
    treatment: str | None = None
    weight_g: float | None = Field(default=None, gt=0)   # one sample unit, in grams
    ingredients: list[IngredientRecord] = Field(default_factory=list)
    experiment_ingredients: list[ExperimentIngredientRecord] = Field(default_factory=list)
    indicators: list[IndicatorRecord] = Field(default_factory=list)
    measurements: list[MeasurementRecord] = Field(default_factory=list)
    evidence: list[EvidenceSpan] = Field(default_factory=list)

class ProtocolRecord(BaseModel):
    """What the model adds to one arm the gate already assembled. Never measurements,
    ingredients or doses — asking for those again only invites them to be retyped.
    `experiment_index` is what joins its answer back to the arm."""

    model_config = ConfigDict(extra="forbid")
    experiment_index: int
    treatment: str | None = None
    weight_g: float | None = Field(default=None, gt=0)
    evidence: list[EvidenceSpan] = Field(default_factory=list)

class AssetHint(BaseModel):
    """What the model may say about an asset the gate could not key: which column orders
    the rows, or that it is not a series at all. Never a value — `adjudicate_review`
    re-reads the table with this hint and the gate extracts the numbers itself."""

    model_config = ConfigDict(extra="ignore")
    docling_item_ref: str
    axis_column: str | None = None
    not_a_series: bool = False
    why: str | None = None

class GoldBundle(BaseModel):
    model_config = ConfigDict(extra="forbid")
    paper: PaperDocument
    sections: list[SectionDocument] = Field(default_factory=list)
    tables: list[TableDocument] = Field(default_factory=list)
    figures: list[FigureDocument] = Field(default_factory=list)
    experiments: list[ExperimentRecord] = Field(default_factory=list)


class OllamaJSONClient:
    """The one model call in the pipeline, to a local Ollama server over its native API.
    Stdlib only: no SDK, no key, nothing to point at a hosted provider by accident."""

    def __init__(self, *, host=OLLAMA_HOST, model=OLLAMA_MODEL):
        self.url = f"{host.rstrip('/')}/api/chat"
        self.model = model

    def json_completion(self, *, system_prompt, user_prompt) -> tuple:
        """Returns (parsed JSON, prompt tokens the server actually read)."""
        body = json.dumps({
            "model": self.model, "stream": False, "format": "json", "think": OLLAMA_THINK,
            "options": {"temperature": 0.0, "num_ctx": OLLAMA_NUM_CTX},
            "messages": [{"role": "system", "content": system_prompt},
                         {"role": "user", "content": user_prompt}],
        }).encode("utf-8")
        request = urllib.request.Request(
            self.url, data=body, headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(request, timeout=OLLAMA_TIMEOUT) as response:
                payload = json.load(response)
        except (urllib.error.URLError, TimeoutError) as exc:
            raise RuntimeError(
                f"Ollama unreachable at {self.url}: {exc}. Start it with `ollama serve`."
            ) from exc
        # `message.thinking` is the model's reasoning channel, not an answer.
        content = (payload.get("message") or {}).get("content") or "{}"
        return json.loads(content), int(payload.get("prompt_eval_count") or 0)


UNATTRIBUTED_MATRIX = "unattributed"

#: How much of a section the model is shown. A methods section states the protocol and
#: the sample mass early; the tail is assay procedure, which neither field needs.
MAX_SECTION_CHARS = int(os.getenv("MAX_SECTION_CHARS", "4000"))


def build_asset_prompt(asset, kind) -> dict:
    gate = asset.get("gate", {})
    all_observations = asset.get("observations", [])
    observations = all_observations[:MAX_PROMPT_OBSERVATIONS]
    return {
        "kind": kind, "docling_item_ref": asset.get("docling_item_ref"),
        "page_number": asset.get("page_number"), "caption": asset.get("caption"),
        "section_hint": asset.get("section_hint"), "axis_label": gate.get("axis_label"),
        "axis_points": gate.get("axis_points"), "value_columns": gate.get("value_columns"),
        "label_columns": gate.get("label_columns"), "orientation": gate.get("orientation"),
        "values_are_approximate": kind == "figure", "observations": observations,
        "observations_truncated": len(all_observations) > len(observations),
        "nearby_text": (asset.get("context_markdown") or "")[:FIGURE_CONTEXT_CHARS],
    }


_WHITESPACE = re.compile(r"\s+")


def _searchable(text) -> str:
    """Case- and whitespace-folded, so a quote is not failed for rewrapped lines."""
    return _WHITESPACE.sub(" ", str(text or "")).strip().lower()


def reference_text_index(package: dict) -> dict:
    """{docling_item_ref: searchable text} over exactly the items the prompt shows, so a
    ref that fails to resolve here is one the model was never given — which is what makes
    the check worth anything. Built once per paper and passed to everything."""
    index = {}
    for section in package.get("sections", []):
        ref = section.get("docling_item_ref")
        if ref:
            index[ref] = _searchable(
                f"{section.get('section_title', '')}\n{section.get('content_markdown', '')}")
    for asset in chain(package.get("tables", []), package.get("references", []),
                       package.get("figures", [])):
        ref = asset.get("docling_item_ref")
        if not ref:
            continue
        parts = [asset.get("caption") or "", asset.get("context_markdown") or "",
                 asset.get("preview_markdown") or ""]
        parts += [f"{item.get('column_label')} {item.get('row_labels')} "
                  f"{item.get('axis_value')} {item.get('value')}"
                  for item in asset.get("observations", [])]
        index[ref] = _searchable("\n".join(parts))
    return index


#: A stated span whose quote is in the item it cites is the only claim that can be
#: checked mechanically, so it is the only one worth full marks. A stated span whose
#: quote is absent scores half: the citation resolved, the wording did not.
STATED_VERBATIM, STATED_UNQUOTED = 1.0, 0.5
METHOD_SCORES = {"derived": 0.7, "inferred": 0.3}
CHART_READING_PENALTY = 0.6


def score_evidence(span, index: dict) -> float:
    """Grade one span. Confidence is an evaluation of it, not a number it chose: a model
    asked for its own reports a mood, whereas whether its quote is in the item it cited
    can be tested. That carries the top of the scale and everything else is placed by
    distance from it, down to zero for a citation resolving to nothing."""
    resolved = index.get(span.docling_item_ref)
    if resolved is None:
        return 0.0
    if span.method == "stated":
        quote = _searchable(span.exact_text)
        score = STATED_VERBATIM if quote and quote in resolved else STATED_UNQUOTED
    else:
        score = METHOD_SCORES[span.method]
    if span.value_is_approximate or span.source_type == "figure":
        score *= CHART_READING_PENALTY
    return round(score, 3)


def score_bundle_evidence(bundle: GoldBundle, index: dict) -> GoldBundle:
    """Overwrite every span's confidence with a computed one. Mutates, and returns."""
    for record in bundle.experiments:
        for span in record.evidence:
            span.confidence = score_evidence(span, index)
    return bundle


def _as_day(axis_value) -> Optional[int]:
    try:
        number = float(axis_value)
    except (TypeError, ValueError):
        return None
    return int(round(number)) if number >= 0 and abs(number - round(number)) <= 1e-6 else None


def _add_ingredients(record, ingredients) -> None:
    known = {item.ingredient_name for item in record.ingredients}
    for item in ingredients:
        if item["name"] in known:
            continue
        known.add(item["name"])
        record.ingredients.append(IngredientRecord(
            ingredient_name=item["name"], functional_class=item["functional_class"],
            source=item["source"]))
        record.experiment_ingredients.append(ExperimentIngredientRecord(
            ingredient_name=item["name"], concentration=item.get("amount"),
            concentration_unit=item.get("unit")))


def build_experiments(measurements, matrix) -> tuple:
    """Group normalised measurements into one record per arm.

    Keyed on `arm_key`, which Silver built from the arm's substances and their amounts:
    nothing else separates one rung of a dose ladder from the next. `treatment` is left
    unset on purpose — it is a protocol, stated in prose the gate never reads. The dedupe
    sets are kept per arm rather than rebuilt from the record on every measurement, and an
    arm's ingredients are added once, since they are the same list on all of them.
    """
    grouped, off_axis = {}, 0
    for measurement in measurements:
        day = _as_day(measurement.axis_value)
        if day is None:
            off_axis += 1
            continue
        state = grouped.get(measurement.arm_key)
        if state is None:
            record = ExperimentRecord(meat_matrix=matrix)
            _add_ingredients(record, measurement.ingredients)
            state = grouped[measurement.arm_key] = (record, set(), set())
        record, seen_indicators, seen_evidence = state

        record.measurements.append(MeasurementRecord(
            day=day, indicator_name=measurement.indicator,
            indicator_type=measurement.indicator_type, indicator_unit=measurement.unit,
            indicator_value=measurement.value, indicator_threshold=measurement.threshold))

        indicator_key = (measurement.indicator, measurement.unit)
        if indicator_key not in seen_indicators:
            seen_indicators.add(indicator_key)
            record.indicators.append(IndicatorRecord(
                indicator_name=measurement.indicator,
                indicator_type=measurement.indicator_type,
                indicator_unit=measurement.unit,
                indicator_threshold=measurement.threshold))

        # One span per (item, field). Deduping on the ref alone would collapse spans that
        # cite the same table for different fields, which is the point of attributing them.
        evidence_key = (measurement.item_ref, "indicator_value")
        if measurement.item_ref and evidence_key not in seen_evidence:
            seen_evidence.add(evidence_key)
            record.evidence.append(EvidenceSpan(
                field_name="indicator_value", docling_item_ref=measurement.item_ref,
                page_number=measurement.page_number,
                source_type="figure" if measurement.is_figure else "table",
                method="stated", value_is_approximate=measurement.is_figure))

    return [record for record, _, _ in grouped.values()], off_axis


def catalogue_experiment(reading, matrix) -> Optional[ExperimentRecord]:
    """Composition-table substances need an experiment row to hang from without
    pretending to be an arm. Its shape says so: ingredients, and not one measurement."""
    if not reading["catalogue"]:
        return None
    record = ExperimentRecord(meat_matrix=matrix)
    _add_ingredients(record, reading["catalogue"])
    return record


def unmeasured_experiment(matrix) -> ExperimentRecord:
    """A paper the gate found no series in still has a matrix and a protocol, and both are
    stated only in prose. Without a row to hang them on they are resolved and then thrown
    away, and the paper is stored as a title and nothing else."""
    return ExperimentRecord(meat_matrix=matrix)


def heuristic_gold_bundle(package: dict) -> GoldBundle:
    """Assemble records from the Silver reading, resolving nothing: every number is fixed
    by the time this returns, and only the two prose-borne fields are missing."""
    reading = package.get("reading") or normalise_silver(package)
    matrix = reading["matrix"]["name"] or UNATTRIBUTED_MATRIX
    experiments, off_axis = build_experiments(reading["measurements"], matrix)
    catalogue = catalogue_experiment(reading, matrix)
    if catalogue:
        experiments.append(catalogue)
    if not experiments:
        experiments.append(unmeasured_experiment(matrix))
        print("        no gated series — one row kept for the matrix and the protocol")

    named = sorted({arm.treatment for arm in reading["lexicon"].values() if arm.treatment})
    print(f"        matrix: {matrix!r} | {len(reading['lexicon'])} arms | "
          f"{len(experiments)} experiments | arm labels name: {', '.join(named) or 'no process'}")
    unclaimed = [item["name"] for item in reading["interventions"] if item["name"] not in named]
    if unclaimed:
        print(f"        prose also names {', '.join(unclaimed)} — no arm label claims it")
    if off_axis:
        print(f"        {off_axis} observations sit off the integer-day axis")

    return GoldBundle(
        paper=PaperDocument(title=Path(package["source_pdf"]).stem.replace("_", " ").title()),
        sections=[SectionDocument.model_validate(s) for s in package.get("sections", [])],
        tables=[TableDocument(caption=t.get("caption"),
                              csv_filepath=t.get("cleaned_csv_path") or t.get("csv_path"),
                              structured_json={"gate": t.get("gate", {}),
                                               "headers": t.get("headers", [])},
                              docling_item_ref=t.get("docling_item_ref"))
                for t in chain(package.get("tables", []), package.get("references", []))],
        figures=[FigureDocument(caption=f.get("caption"), image_filepath=f["image_path"],
                                docling_item_ref=f.get("docling_item_ref"))
                 for f in package.get("figures", [])],
        experiments=experiments,
    )


# Two fields and their attribution, nothing else: the matrix, the arms, the doses and
# every measurement are resolved before the call, so there is no number here to retype.
GOLD_SYSTEM_PROMPT = """\
You are a food-science extraction assistant reading the METHODS of one paper.

Each experimental arm below is already resolved: its matrix, its ingredients, their
concentrations, its indicators and every measured value. Do not restate any of them.
Return exactly two things per arm, plus the evidence for each.

## treatment

A concise protocol summary, in prose, of what happened to THIS arm. Cover only what
the paper says:
  * how the ingredient was applied — coating, dipping, immersion, mixing, injection,
    spraying, marination, surface application
  * processing performed — chilling, freezing, heating, cooking, drying, irradiation,
    washing
  * packaging used for this arm
  * storage conditions that are part of the protocol
  * procedural detail — temperatures, durations, repeat counts, drain periods,
    incubation times, order of operations

Never include ingredient names, concentrations, dose ladders, treatment-group labels,
or control-group descriptions. Those live in `ingredients` and
`experiment_ingredients.concentration`, and repeating them here duplicates the schema.

GOOD: "Chicken fillets were coated by dipping into the coating dispersion. Two dipping
cycles of 120 s each were performed with a 2 min draining period between applications
before packaging in polystyrene trays and refrigerated storage."
BAD:  "Samples received 0%, 1%, and 2% thyme oil coatings."   <- doses, not a protocol

Set `treatment` to null ONLY when the paper says this arm was stored or observed with
no treatment or intervention, and emit evidence saying so. If the paper simply never
describes this arm's protocol, OMIT the field — that is missing, which is not the same
fact, and inventing a protocol to fill it is the worst thing you can do here.

## weight_g

The mass of ONE experimental sample unit — a meatball, fillet, fillet cube, steak,
patty, sausage, shrimp portion — converted to GRAMS. In priority order:
  1. a directly stated value
  2. a value from arithmetic the paper supplies both operands for
  3. a conservative bound the paper supports
  4. otherwise null

Never derive it from coating volume, inoculum volume, an analytical or extraction
subsample size, typical values from the literature, or what you know about food
portions. If nothing in the paper constrains it, return null and say so in evidence.

## evidence

One object per field you return, including fields you return as null:

{
  "field_name"   : "treatment" | "weight_g",
  "docling_item_ref": REQUIRED — copy one from the items you were shown; never invent one,
  "page_number"  : integer or null,
  "source_type"  : "prose" | "table" | "figure",
  "source_label" : "Table 2" | "Section 2.3" | null,
  "exact_text"   : the verbatim sentence — REQUIRED when method is "stated" and
                   source_type is "prose",
  "method"       : "stated" | "derived" | "inferred",
  "rationale"    : REQUIRED when method is "derived" or "inferred" — the arithmetic or
                   the reasoning, named explicitly, including how weak it is
}

Do NOT return a confidence. It is computed from your citation, by checking your quote
against the item you cited; a number you supply would be ignored.

Worked examples:

  stated   — weight_g 25, exact_text "The artificially inoculated ground beef was
             formed into meatballs (25 g each)."
  derived  — weight_g 50, rationale "The paper states a 1 kg batch was divided into 20
             equal patties (1000 g / 20 = 50 g each)."
  inferred — weight_g null, rationale "The paper states fillet cubes were approximately
             equal in size but never reports the mass of one. Nothing in the paper
             constrains it, so no value is given."

Return ONLY valid JSON:
{"experiments": [{"experiment_index": 0, "treatment": "...", "weight_g": 25.0,
                  "evidence": [ ... ]}]}
"""


def _arm_prompt(bundle: GoldBundle) -> list:
    """The arms as the model sees them, each under the index that joins its reply back.
    Doses are shown so it can tell the arms apart, not so it can return them."""
    return [{
        "experiment_index": index,
        "meat_matrix": record.meat_matrix,
        "ingredients": [{"ingredient_name": link.ingredient_name,
                         "concentration": link.concentration,
                         "concentration_unit": link.concentration_unit}
                        for link in record.experiment_ingredients],
        "days": sorted({m.day for m in record.measurements}),
        "indicators": sorted({m.indicator_name for m in record.measurements}),
    } for index, record in enumerate(bundle.experiments)]


def apply_protocols(bundle: GoldBundle, payload: dict) -> GoldBundle:
    """Merge the model's reply into the assembled bundle. Mutates, and returns.

    Each arm is validated on its own, so a malformed entry costs that arm and not the
    paper; an index outside the bundle means an arm was invented and is dropped too. A
    paper whose every arm was dropped is reported by `validate_bundle_semantics`.
    """
    entries = payload.get("experiments")
    if not isinstance(entries, list):
        print("        protocol reply has no experiments list — nothing applied")
        return bundle
    for position, entry in enumerate(entries):
        try:
            record = ProtocolRecord.model_validate(entry)
        except ValidationError as exc:
            print(f"        protocol entry {position} does not fit the schema — dropped "
                  f"({exc.error_count()} problems, first: {exc.errors()[0]['msg'][:60]})")
            continue
        if not 0 <= record.experiment_index < len(bundle.experiments):
            print(f"        protocol names experiment {record.experiment_index}, which "
                  f"the gate did not find — dropped")
            continue
        experiment = bundle.experiments[record.experiment_index]
        experiment.treatment = record.treatment
        experiment.weight_g = record.weight_g
        experiment.evidence.extend(record.evidence)
    return bundle


def build_gold_from_silver(package: dict, client, index: dict) -> GoldBundle:
    """Assemble from Silver, then ask the local model only for what prose alone carries.
    No fallback to the bare bundle: an arm with no protocol and no mass would look
    exactly like a paper that described neither."""
    bundle = heuristic_gold_bundle(package)
    user_prompt = json.dumps({
        "paper_slug": package["paper_slug"],
        "sections": [{"docling_item_ref": s.get("docling_item_ref"),
                      "section_title": s["section_title"],
                      "content_markdown": s["content_markdown"][:MAX_SECTION_CHARS]}
                     for s in package.get("sections", [])],
        "arms": _arm_prompt(bundle),
        "tables": [build_asset_prompt(t, "table") for t in package.get("tables", [])],
        "figures": [build_asset_prompt(f, "figure") for f in package.get("figures", [])],
        "reference_tables": [{"caption": r.get("caption"), "headers": r.get("headers"),
                              "docling_item_ref": r.get("docling_item_ref")}
                             for r in package.get("references", [])],
        "valid_docling_item_refs": sorted(index),
    }, ensure_ascii=False, indent=2, default=str)

    # Too small a context truncates instead of complaining, and the head of the prompt —
    # the methods prose — is what goes. The estimate warns before the call;
    # `prompt_eval_count` is what the server actually read.
    estimate = (len(GOLD_SYSTEM_PROMPT) + len(user_prompt)) // 4
    print(f"        gold prompt: ~{estimate} tokens over {len(bundle.experiments)} arms")

    payload, prompt_tokens = client.json_completion(
        system_prompt=GOLD_SYSTEM_PROMPT, user_prompt=user_prompt)
    if prompt_tokens:
        margin = "TRUNCATED" if prompt_tokens >= OLLAMA_NUM_CTX else "ok"
        print(f"        ollama read {prompt_tokens}/{OLLAMA_NUM_CTX} context tokens ({margin})")
    apply_protocols(bundle, payload)
    return score_bundle_evidence(bundle, index)

## Validation and Report

Cross-record rules the per-record models cannot express, plus what the vocabulary changed, discarded, or could not name.

In [8]:
VIOLATION_COLUMNS = ["rule", "table", "key", "detail"]

#: Protocol prose runs to a couple of sentences, and a violation frame is unreadable with
#: one wrapped into every key.
SUMMARY_CHARS = 60


def _short(text) -> str:
    text = " ".join(str(text or "").split())
    return text if len(text) <= SUMMARY_CHARS else text[:SUMMARY_CHARS - 1] + "…"


def _check_treatment(record, key, substances):
    """`treatment` is free prose, so no vocabulary applies. What is still checkable is
    that it is a protocol and not one of the things the schema keeps elsewhere."""
    if record.treatment is None:
        return
    if canonical_key(record.treatment) in substances:
        yield "treatment_must_not_be_an_ingredient", "experiments", key, \
            f"{_short(record.treatment)!r} is also an ingredient name"
    if parse_dosed_label(record.treatment):
        yield "dose_belongs_in_concentration", "experiments", key, \
            f"{_short(record.treatment)!r} carries an amount — it belongs in concentration"


def _check_attribution(record, key, index):
    """No value without a source. A null field is deliberately exempt: absence of a span
    is how "never addressed" is recorded, and demanding one would erase the difference
    from "the paper says there was none"."""
    attributed = {span.field_name for span in record.evidence}
    for field_name in ("treatment", "weight_g"):
        if getattr(record, field_name) is not None and field_name not in attributed:
            yield f"{field_name}_needs_evidence", "experiments", key, \
                f"{field_name} is set, but no evidence span says where it came from"
    if index is None:
        return
    for span in record.evidence:
        if span.docling_item_ref not in index:
            yield "evidence_ref_must_resolve", "evidence", f"{key}/{span.field_name}", \
                f"{span.docling_item_ref!r} is not an item the model was shown"


def _check_ingredients(record, key):
    declared = {i.ingredient_name for i in record.ingredients}
    for item in record.ingredients:
        if VOCABULARY.resolve(item.ingredient_name, "ingredient") is None:
            yield "ingredient_must_be_in_vocabulary", "ingredients", item.ingredient_name, \
                "not a known substance — a table read as a catalogue by shape alone"
        if item.functional_class == UNCLASSIFIED_CLASS:
            yield "ingredient_needs_a_functional_class", "ingredients", item.ingredient_name, \
                "unclassified — add functional_class to vocabulary.json"
        if item.source == UNKNOWN_SOURCE:
            yield "ingredient_needs_a_source", "ingredients", item.ingredient_name, \
                "unknown origin — add source to vocabulary.json"
    for link in record.experiment_ingredients:
        if link.ingredient_name not in declared:
            yield "link_must_reference_a_declared_ingredient", "experiment_ingredients", \
                f"{key}/{link.ingredient_name}", "no matching ingredient record"
        elif link.concentration is None:
            yield "link_needs_a_concentration", "experiment_ingredients", \
                f"{key}/{link.ingredient_name}", "no amount was read for this arm"


def _check_indicators(record, key):
    declared = {(i.indicator_name, i.indicator_unit) for i in record.indicators}
    for item in record.indicators:
        if _DISCARDED.search(canonical_key(item.indicator_name)):
            yield "indicator_must_not_be_sensory", "indicators", item.indicator_name, \
                "sensory or gravimetric — should have been dropped in Silver"
        if item.indicator_name == UNRESOLVED_INDICATOR:
            yield "indicator_needs_a_name", "indicators", f"{key}/{item.indicator_unit}", \
                "no caption or label named the quantity"
        if item.indicator_unit == UNSPECIFIED_UNIT:
            yield "indicator_needs_a_unit", "indicators", f"{key}/{item.indicator_name}", \
                "no unit in the label and none in the vocabulary"
    for item in record.measurements:
        if (item.indicator_name, item.indicator_unit) not in declared:
            yield "measurement_must_reference_a_declared_indicator", "measurements", \
                f"{key}/day {item.day}/{item.indicator_name}", "indicator not listed on this arm"


def validate_bundle_semantics(bundle: GoldBundle, index: dict | None = None) -> pd.DataFrame:
    """Cross-record rules the per-record models cannot express. Never raises: one pass
    shows everything wrong, and an empty frame means the bundle is consistent. `index` is
    what a citation resolves against; without it the ref rule is skipped, not guessed."""
    substances = {canonical_key(i.ingredient_name)
                  for record in bundle.experiments for i in record.ingredients}
    rows = []
    for record in bundle.experiments:
        key = (f"{record.meat_matrix}/{_short(record.treatment) or 'no protocol'}/"
               + (", ".join(sorted(i.ingredient_name for i in record.ingredients)) or "control"))
        rows += list(_check_treatment(record, key, substances))
        rows += list(_check_attribution(record, key, index))
        rows += list(_check_ingredients(record, key))
        rows += list(_check_indicators(record, key))
    return pd.DataFrame(rows, columns=VIOLATION_COLUMNS)


def _changed_frame(vocabulary) -> pd.DataFrame:
    """`renamed` separates a value changed ("TVB-N" -> "Total volatile basic nitrogen")
    from one merely confirmed ("pH" -> "pH")."""
    columns = ["kind", "raw", "canonical", "renamed", "functional_class", "source",
               "indicator_type", "unit"]
    frame = pd.DataFrame([{
        "kind": r["kind"], "raw": r["raw"], "canonical": r["canonical"],
        "renamed": canonical_key(r["raw"]) != canonical_key(r["canonical"]),
        "functional_class": r.get("functional_class"), "source": r.get("source"),
        "indicator_type": r.get("indicator_type"), "unit": r.get("unit"),
    } for r in vocabulary.changes.values()], columns=columns)
    return frame.sort_values(["kind", "canonical"], ignore_index=True) if not frame.empty else frame


def _vocabulary_frame() -> pd.DataFrame:
    return pd.DataFrame(
        [{"field": "indicators.indicator_type", "value": v, "rolls_up_to": ""}
         for v in INDICATOR_TYPES]
        + [{"field": "ingredients.functional_class", "value": leaf, "rolls_up_to": tier}
           for leaf, tier in FUNCTIONAL_CLASS_TIERS.items()]
        + [{"field": "ingredients.source", "value": v, "rolls_up_to": ""}
           for v in INGREDIENT_SOURCES]
        + [{"field": "evidence.method", "value": v, "rolls_up_to": ""}
           for v in EVIDENCE_METHODS]
        + [{"field": "evidence.source_type", "value": v, "rolls_up_to": ""}
           for v in EVIDENCE_SOURCE_TYPES])


def _review_frame(vocabulary, violations) -> pd.DataFrame:
    """A violation names the same value as the flag behind it and says more, so it wins.
    A discarded value is already reported, with its count, in `discarded`."""
    rows = []
    covered = {canonical_key(value) for _, value in vocabulary.discarded}

    def take(kind, value, reason):
        if canonical_key(value) not in covered:
            covered.add(canonical_key(value))
            rows.append({"kind": kind, "value": value, "reason": reason})

    if violations is not None and not violations.empty:
        for _, violation in violations.iterrows():
            take(violation["table"], violation["key"],
                 f"{violation['rule']}: {violation['detail']}")
    for (kind, value), reason in vocabulary.flags.items():
        take(kind, value, reason)
    for kind, names in vocabulary.unresolved.items():
        for name in names.values():
            take(kind, name, "not in vocabulary — kept verbatim")
    return pd.DataFrame(rows, columns=["kind", "value", "reason"])


def _discarded_frame(vocabulary) -> pd.DataFrame:
    """What never reached the database, and why. A drop nobody can see is a bug."""
    return pd.DataFrame(
        [{"kind": kind, "value": value, "occurrences": count,
          "reason": "sensory or gravimetric — not a shelf-life indicator" if kind == "indicator"
                    else "not a known substance — add it to vocabulary.json to keep it"}
         for (kind, value), count in sorted(vocabulary.discarded.items())],
        columns=["kind", "value", "occurrences", "reason"])


def normalization_report(violations: pd.DataFrame | None = None, vocabulary=None) -> dict:
    vocabulary = vocabulary or VOCABULARY
    return {"changed": _changed_frame(vocabulary), "vocabulary": _vocabulary_frame(),
            "discarded": _discarded_frame(vocabulary),
            "review": _review_frame(vocabulary, violations)}


def show_normalization_report(violations: pd.DataFrame | None = None) -> dict:
    report = normalization_report(violations)
    changed, review, discarded = report["changed"], report["review"], report["discarded"]
    print("─ normalisation " + "─" * 62)
    for kind, group in changed.groupby("kind") if not changed.empty else ():
        print(f"  {kind:<11}: {len(group):>3} raw -> {group['canonical'].nunique():>3} canonical  "
              f"({int(group['renamed'].sum())} rewritten, "
              f"{len(group) - group['canonical'].nunique()} merged)")
    print(f"  values changed         : {0 if changed.empty else int(changed['renamed'].sum())}")
    print(f"  controlled values      : {len(report['vocabulary'])}")
    for kind, group in discarded.groupby("kind") if not discarded.empty else ():
        print(f"  {kind + ' discarded':<22}: {len(group):>3} names, "
              f"{int(group['occurrences'].sum())} occurrences")
    print(f"  needing manual review  : {len(review)}")
    for reason, group in review.groupby("reason") if not review.empty else ():
        print(f"      {len(group):>3}  {reason[:82]}")
    print("─" * 78)
    for title, frame in (("changed", changed), ("vocabulary", report["vocabulary"]),
                         ("discarded", discarded), ("review", review)):
        if not frame.empty:
            print(f"\n{title}:")
            display(frame)
    return report

## Local SQLite Schema

Mirrors the production layout. The vocabularies are `CHECK` constraints here too, so a bad value fails on write.

In [9]:
from sqlalchemy import (
    Boolean, CheckConstraint, Column, DateTime, Float, ForeignKey, Integer, String,
    Text, UniqueConstraint, create_engine, inspect,
)
from sqlalchemy.orm import declarative_base, sessionmaker


def _sql_values(values):
    """Keeps the CHECK constraints and the Python vocabularies from drifting apart."""
    return ", ".join("'" + str(value).replace("'", "''") + "'" for value in values)


Base = declarative_base()
ENGINE = create_engine(f"sqlite:///{LOCAL_DB_PATH}", future=True)
SessionLocal = sessionmaker(bind=ENGINE, autoflush=False, autocommit=False, future=True)

class PaperRow(Base):
    __tablename__ = "papers"

    id = Column(Integer, primary_key=True, index=True)
    doi = Column(String, nullable=True, index=True)
    title = Column(String, nullable=False)
    abstract = Column(Text, nullable=True)
    published_year = Column(Integer, nullable=True)
    source_path = Column(String, nullable=False)
    file_hash = Column(String(64), unique=True, nullable=False, index=True)
    bronze_path = Column(String, nullable=False)
    silver_path = Column(String, nullable=True)
    markdown_path = Column(String, nullable=True)
    created_at = Column(DateTime, default=datetime.utcnow)

class SectionRow(Base):
    __tablename__ = "sections"

    id = Column(Integer, primary_key=True, index=True)
    paper_id = Column(Integer, ForeignKey("papers.id", ondelete="CASCADE"), nullable=False, index=True)
    section_title = Column(String, nullable=False)
    content_markdown = Column(Text, nullable=False)
    embedding = Column(Text, nullable=True)
    section_order = Column(Integer, nullable=True)
    docling_item_ref = Column(String, nullable=True)   # what an evidence row cites

class TableRow(Base):
    __tablename__ = "tables"

    id = Column(Integer, primary_key=True, index=True)
    paper_id = Column(Integer, ForeignKey("papers.id", ondelete="CASCADE"), nullable=False, index=True)
    caption = Column(Text, nullable=True)
    csv_filepath = Column(String, nullable=False)
    structured_json = Column(Text, nullable=True)
    docling_item_ref = Column(String, nullable=True)
    created_at = Column(DateTime, default=datetime.utcnow)

class FigureRow(Base):
    __tablename__ = "figures"

    id = Column(Integer, primary_key=True, index=True)
    paper_id = Column(Integer, ForeignKey("papers.id", ondelete="CASCADE"), nullable=False, index=True)
    caption = Column(Text, nullable=True)
    image_filepath = Column(String, nullable=False)
    semantic_tags_json = Column(Text, nullable=True)
    docling_item_ref = Column(String, nullable=True)
    created_at = Column(DateTime, default=datetime.utcnow)

class ExperimentRow(Base):
    """One arm. `treatment` is its protocol in prose, the additive dimension is
    `experiment_ingredients`, and a control has no rows there. Nothing here identifies an
    arm on its own, hence no unique constraint. Both nullable columns are null for two
    reasons that only a join to `evidence` tells apart."""

    __tablename__ = "experiments"

    experiment_id = Column(Integer, primary_key=True, index=True)
    meat_matrix = Column(String, nullable=False)
    treatment = Column(String, nullable=True)
    weight_g = Column(Float, nullable=True)      # one sample unit, normalised to grams
    __table_args__ = (CheckConstraint("weight_g IS NULL OR weight_g > 0",
                                      name="ck_experiments_weight_positive"),)

class IngredientRow(Base):
    __tablename__ = "ingredients"

    ingredient_id = Column(Integer, primary_key=True, index=True)
    ingredient_name = Column(String, nullable=False, unique=True)
    functional_class = Column(String, nullable=False)
    source = Column(String, nullable=False)   # origin, not a document reference
    __table_args__ = (
        CheckConstraint(f"functional_class IN ({_sql_values(FUNCTIONAL_CLASSES + (UNCLASSIFIED_CLASS,))})",
                        name="ck_ingredients_functional_class"),
        CheckConstraint(f"source IN ({_sql_values(INGREDIENT_SOURCES + (UNKNOWN_SOURCE,))})",
                        name="ck_ingredients_source"),
    )

class ExperimentIngredientRow(Base):
    """`concentration` is where every amount lives."""

    __tablename__ = "experiment_ingredients"

    experiment_id = Column(Integer, ForeignKey("experiments.experiment_id", ondelete="CASCADE"), primary_key=True)
    ingredient_id = Column(Integer, ForeignKey("ingredients.ingredient_id", ondelete="CASCADE"), primary_key=True)
    concentration = Column(Float, nullable=True)
    concentration_unit = Column(String, nullable=True)

class IndicatorRow(Base):
    """Unique on (name, unit): `indicator_type` is a two-valued category, so keying on it
    would fold every microbial count in log CFU/g into one row."""

    __tablename__ = "indicators"

    indicator_id = Column(Integer, primary_key=True, index=True)
    indicator_name = Column(String, nullable=False)
    indicator_type = Column(String, nullable=False)
    indicator_unit = Column(String, nullable=False)
    indicator_threshold = Column(Float, nullable=True)
    __table_args__ = (
        UniqueConstraint("indicator_name", "indicator_unit", name="uq_indicator_name_unit"),
        CheckConstraint(f"indicator_type IN ({_sql_values(INDICATOR_TYPES)})",
                        name="ck_indicators_type"),
    )

class MeasurementRow(Base):
    __tablename__ = "measurements"

    experiment_id = Column(Integer, ForeignKey("experiments.experiment_id", ondelete="CASCADE"), primary_key=True)
    day = Column(Integer, primary_key=True)
    indicator_id = Column(Integer, ForeignKey("indicators.indicator_id", ondelete="CASCADE"), primary_key=True)
    indicator_value = Column(Float, nullable=False)
    __table_args__ = (CheckConstraint("day >= 0", name="ck_measurements_day_non_negative"),)

class EvidenceRow(Base):
    """What supports one extracted field. `field_name` makes it attributable; `method`
    and `rationale` say how far it sits from the paper's own words. `confidence` is
    computed by `score_evidence`, never taken from the model."""

    __tablename__ = "evidence"

    id = Column(Integer, primary_key=True, index=True)
    paper_id = Column(Integer, ForeignKey("papers.id", ondelete="CASCADE"), nullable=False, index=True)
    entity_type = Column(String, nullable=False)
    entity_key = Column(Text, nullable=False)
    field_name = Column(String, nullable=False)   # the field this row supports
    page_number = Column(Integer, nullable=True)
    source_type = Column(String, nullable=False)
    source_label = Column(String, nullable=True)
    exact_text = Column(Text, nullable=True)
    method = Column(String, nullable=False)       # stated | derived | inferred
    rationale = Column(Text, nullable=True)       # required for derived and inferred
    confidence = Column(Float, nullable=True)
    value_is_approximate = Column(Boolean, default=False)
    docling_item_ref = Column(String, nullable=True)
    __table_args__ = (
        CheckConstraint(f"method IN ({_sql_values(EVIDENCE_METHODS)})",
                        name="ck_evidence_method"),
        CheckConstraint(f"source_type IN ({_sql_values(EVIDENCE_SOURCE_TYPES)})",
                        name="ck_evidence_source_type"),
        CheckConstraint("method = 'stated' OR (rationale IS NOT NULL AND rationale != '')",
                        name="ck_evidence_rationale_when_not_stated"),
    )


# create_all creates, never alters, so a database written against an older column set
# looks fine and fails on the first insert. Moved aside, never dropped.
_REQUIRED_COLUMNS = {"indicators": {"indicator_name"},
                     "sections": {"docling_item_ref"},
                     "experiments": {"weight_g"},
                     "evidence": {"method", "rationale"}}
_REMOVED_COLUMNS = {"experiments": {"arm_label"}, "measurements": {"value_is_approximate"},
                    "papers": {"docling_json_path"}}


def init_local_database() -> None:
    inspector = inspect(ENGINE)
    existing = set(inspector.get_table_names())

    def columns(table):
        return {column["name"] for column in inspector.get_columns(table)}

    stale = [f"{table} is missing {', '.join(sorted(required - columns(table)))}"
             for table, required in _REQUIRED_COLUMNS.items()
             if table in existing and required - columns(table)]
    stale += [f"{table} still has {', '.join(sorted(gone & columns(table)))}"
              for table, gone in _REMOVED_COLUMNS.items()
              if table in existing and gone & columns(table)]
    if stale:
        backup = LOCAL_DB_PATH.with_name(LOCAL_DB_PATH.name + ".pre-normalisation")
        ENGINE.dispose()
        shutil.move(str(LOCAL_DB_PATH), str(backup))
        print("Stale local database (" + "; ".join(stale) + f") moved to {backup.name}")
    Base.metadata.create_all(ENGINE)


def _json_text(payload):
    return None if payload is None else json.dumps(payload, ensure_ascii=False, default=str)


def get_or_create_paper(session, manifest, bundle) -> PaperRow:
    paper = session.query(PaperRow).filter(PaperRow.file_hash == manifest["file_hash"]).one_or_none()
    if paper is None:
        paper = PaperRow(
            doi=bundle.paper.doi,
            title=bundle.paper.title,
            abstract=bundle.paper.abstract,
            published_year=bundle.paper.published_year,
            source_path=manifest["source_pdf"], file_hash=manifest["file_hash"],
            bronze_path=str(Path(manifest["markdown_path"]).parent),
            silver_path=str(SILVER_ROOT / manifest["paper_slug"]),
            markdown_path=manifest["markdown_path"])
        session.add(paper)
        session.flush()
    return paper


def persist_sections(session, paper_id, sections) -> None:
    for order, section in enumerate(sections, start=1):
        session.add(SectionRow(paper_id=paper_id, section_order=order,
                               section_title=section.section_title,
                               content_markdown=section.content_markdown,
                               embedding=section.embedding,
                               docling_item_ref=section.docling_item_ref))


def persist_tables(session, paper_id, tables) -> None:
    for table in tables:
        session.add(TableRow(paper_id=paper_id, caption=table.caption,
                             csv_filepath=table.csv_filepath,
                             structured_json=_json_text(table.structured_json),
                             docling_item_ref=table.docling_item_ref))


def persist_figures(session, paper_id, figures) -> None:
    for figure in figures:
        session.add(FigureRow(paper_id=paper_id, caption=figure.caption,
                              image_filepath=figure.image_filepath,
                              semantic_tags_json=_json_text(figure.semantic_tags),
                              docling_item_ref=figure.docling_item_ref))


def _link_signature(links):
    return sorted((l.ingredient_name, l.concentration, l.concentration_unit) for l in links)


def _stored_signatures(session, meat_matrix, treatment) -> list:
    """Every stored arm with this matrix and protocol, each with its ingredient
    signature. Two queries for the whole group, not a JOIN per candidate row."""
    rows = session.query(ExperimentRow).filter_by(
        meat_matrix=meat_matrix, treatment=treatment).all()
    if not rows:
        return []
    by_experiment = {row.experiment_id: [] for row in rows}
    links = (session.query(ExperimentIngredientRow.experiment_id,
                           IngredientRow.ingredient_name,
                           ExperimentIngredientRow.concentration,
                           ExperimentIngredientRow.concentration_unit)
             .join(IngredientRow,
                   IngredientRow.ingredient_id == ExperimentIngredientRow.ingredient_id)
             .filter(ExperimentIngredientRow.experiment_id.in_(list(by_experiment))))
    for experiment_id, name, concentration, unit in links:
        by_experiment[experiment_id].append((name, concentration, unit))
    return [[row, sorted(by_experiment[row.experiment_id])] for row in rows]


def _get_or_create_experiment(session, record, signatures) -> ExperimentRow:
    """Matched on matrix, treatment and the arm's ingredients with their amounts — the
    whole identity of an arm, since two rungs of a dose ladder differ only in
    `concentration` and a control is the arm whose set is empty. `weight_g` is not part
    of it: a re-read mass would otherwise split one arm in two."""
    key = (record.meat_matrix, record.treatment)
    stored = signatures.get(key)
    if stored is None:
        stored = signatures[key] = _stored_signatures(session, *key)
    want = _link_signature(record.experiment_ingredients)
    for row, signature in stored:
        if signature == want:
            return row
    row = ExperimentRow(meat_matrix=record.meat_matrix, treatment=record.treatment,
                        weight_g=record.weight_g)
    session.add(row)
    session.flush()
    stored.append([row, want])
    return row


def _get_or_create_ingredient(session, record, cache) -> IngredientRow:
    row = cache.get(record.ingredient_name)
    if row is None:
        row = session.query(IngredientRow).filter_by(
            ingredient_name=record.ingredient_name).one_or_none()
        if row is None:
            row = IngredientRow(ingredient_name=record.ingredient_name,
                                functional_class=record.functional_class, source=record.source)
            session.add(row)
            session.flush()
        cache[record.ingredient_name] = row
    return row


def _get_or_create_indicator(session, record, cache) -> IndicatorRow:
    key = (record.indicator_name, record.indicator_unit)
    row = cache.get(key)
    if row is None:
        row = session.query(IndicatorRow).filter_by(
            indicator_name=record.indicator_name,
            indicator_unit=record.indicator_unit).one_or_none()
        if row is None:
            row = IndicatorRow(indicator_name=record.indicator_name,
                               indicator_type=record.indicator_type,
                               indicator_unit=record.indicator_unit,
                               indicator_threshold=record.indicator_threshold)
            session.add(row)
            session.flush()
        cache[key] = row
    if record.indicator_threshold is not None and row.indicator_threshold is None:
        row.indicator_threshold = record.indicator_threshold
    return row


def persist_gold_bundle(session, paper_id, bundle) -> dict:
    """One pass over the bundle. The three caches are what keep this from issuing a
    SELECT per measurement and a JOIN per stored arm."""
    counts = {"experiments": 0, "ingredients": 0, "indicators": 0, "measurements": 0,
              "measurement_collisions": 0, "evidence": 0}
    seen = set()
    signatures, ingredients, indicators = {}, {}, {}
    for record in bundle.experiments:
        experiment = _get_or_create_experiment(session, record, signatures)
        counts["experiments"] += 1
        for item in record.ingredients:
            _get_or_create_ingredient(session, item, ingredients)
            counts["ingredients"] += 1
        for link in record.experiment_ingredients:
            ingredient = ingredients.get(link.ingredient_name)
            if ingredient is None:
                ingredient = session.query(IngredientRow).filter_by(
                    ingredient_name=link.ingredient_name).one_or_none()
            if ingredient is not None:
                session.merge(ExperimentIngredientRow(
                    experiment_id=experiment.experiment_id, ingredient_id=ingredient.ingredient_id,
                    concentration=link.concentration, concentration_unit=link.concentration_unit))
        for item in record.indicators:
            _get_or_create_indicator(session, item, indicators)
            counts["indicators"] += 1
        for item in record.measurements:
            indicator = _get_or_create_indicator(session, item, indicators)
            # One value per (experiment, day, indicator). Replicates and two columns
            # sharing a label land on the same key; the first wins and the rest are
            # counted, because averaging would invent a number the paper never printed.
            key = (experiment.experiment_id, item.day, indicator.indicator_id)
            if key in seen:
                counts["measurement_collisions"] += 1
                continue
            seen.add(key)
            session.merge(MeasurementRow(
                experiment_id=experiment.experiment_id, day=item.day,
                indicator_id=indicator.indicator_id, indicator_value=item.indicator_value))
            counts["measurements"] += 1
        # Every span is arm-level: the model answers per arm, and a gate span cites a
        # whole table. Inventing a (day, indicator) key would claim a precision the
        # citation does not have.
        for item in record.evidence:
            session.add(EvidenceRow(
                paper_id=paper_id, entity_type="experiment",
                entity_key=_json_text({"experiment_id": experiment.experiment_id}),
                field_name=item.field_name,
                page_number=item.page_number, source_type=item.source_type,
                source_label=item.source_label, exact_text=item.exact_text,
                method=item.method, rationale=item.rationale,
                confidence=item.confidence, value_is_approximate=item.value_is_approximate,
                docling_item_ref=item.docling_item_ref))
            counts["evidence"] += 1
    return counts


init_local_database()
print(f"SQLite schema ready at {LOCAL_DB_PATH}")

SQLite schema ready at /content/medallion_extraction/extraction_local.sqlite


## Runner

Bronze → Silver → normalise → Gold → SQLite. Start Ollama, set `PAPER_INPUTS`, and run.

In [10]:
def build_ollama_client() -> OllamaJSONClient:
    """Check the server is up and the model is pulled, then hand back the client — here
    rather than at the first Gold call, so a stopped server costs nothing instead of
    surfacing after Docling has spent minutes on the first paper."""
    tags_url = f"{OLLAMA_HOST.rstrip('/')}/api/tags"
    try:
        with urllib.request.urlopen(tags_url, timeout=10) as response:
            entries = json.load(response).get("models", [])
    except (urllib.error.URLError, TimeoutError, ValueError) as exc:
        raise RuntimeError(
            f"No Ollama server at {OLLAMA_HOST} ({exc}). Gold needs one to read "
            "`treatment` and `weight_g` out of the methods prose — start it with "
            "`ollama serve`, or point OLLAMA_HOST at another host.") from exc

    available = {name for name in (entry.get("model") or entry.get("name")
                                   for entry in entries) if name}
    if OLLAMA_MODEL not in available:
        raise RuntimeError(
            f"{OLLAMA_MODEL!r} is not pulled on {OLLAMA_HOST}. Run "
            f"`ollama pull {OLLAMA_MODEL}`. Available: {', '.join(sorted(available)) or 'none'}")
    return OllamaJSONClient()


def persist_document(session, manifest, bundle) -> dict:
    paper = get_or_create_paper(session, manifest, bundle)
    persist_sections(session, paper.id, bundle.sections)
    persist_tables(session, paper.id, bundle.tables)
    persist_figures(session, paper.id, bundle.figures)
    counts = persist_gold_bundle(session, paper.id, bundle)
    session.commit()
    return {"paper_id": paper.id, **counts, "sections": len(bundle.sections),
            "tables": len(bundle.tables), "figures": len(bundle.figures)}


def run_pipeline_for_pdf(pdf_path, *, llm_client, show_decisions=False) -> tuple:
    """One paper, end to end. Returns (summary row, violation frame)."""
    manifest = extract_bronze(Path(pdf_path))
    package = build_silver_package(manifest)
    # Still Silver: the model names the axis of anything the gate could not key, and the
    # gate re-reads those tables itself. Before normalisation, so a promoted asset is
    # interpreted by exactly the same path as one that keyed on its own.
    adjudicate_review(package, llm_client)
    if show_decisions:
        show_gate_decisions(package)
    normalise_silver(package)                    # Silver: everything is resolved here

    # Built once and shared: the prompt lists these refs, the evidence scorer grades
    # against them, and the ref rule checks them.
    index = reference_text_index(package)
    bundle = build_gold_from_silver(package, client=llm_client, index=index)

    violations = validate_bundle_semantics(bundle, index)
    if not violations.empty:
        # Reported, not raised: the paper's good rows are still worth keeping.
        print(f"        {len(violations)} semantic violations — see show_normalization_report()")

    with SessionLocal() as session:
        counts = persist_document(session, manifest, bundle)

    report = package["gate_report"]
    return {"pdf_path": str(pdf_path), "paper_slug": manifest["paper_slug"],
            "native_pdf": manifest["native_pdf"], "bronze_seconds": manifest["elapsed_seconds"],
            "bronze_tables": report["tables_in"], "bronze_figures": report["figures_in"],
            "gated_tables": report["tables_accepted"], "gated_figures": report["figures_accepted"],
            "reference_assets": report["references"], "observations": report["observations"],
            "review_open": report.get("review", 0),
            "review_keyed": report.get("review_promoted", 0),
            "gold_experiments": len(bundle.experiments), "violations": len(violations),
            **counts}, violations


def run_pipeline(pdf_paths, *, show_decisions=True, show_report=True):
    """Bronze -> Silver -> Gold over each PDF. The Ollama client is built first, so a
    server that is not running costs nothing."""
    client = build_ollama_client()
    summaries, violations = [], []
    for path in pdf_paths:
        summary, found = run_pipeline_for_pdf(path, llm_client=client,
                                              show_decisions=show_decisions)
        summaries.append(summary)
        violations.append(found)
    frame = pd.DataFrame(summaries)
    display(frame)
    if show_report:
        show_normalization_report(pd.concat(violations, ignore_index=True)
                                  if violations else None)
    return frame


PAPER_INPUTS = ["11.pdf", "12.pdf", "13.pdf"]

if PAPER_INPUTS:
    summary_df = run_pipeline(PAPER_INPUTS)
else:
    print("Set PAPER_INPUTS to one or more uploaded PDF files, then run this cell again.")

Bronze: 11 | native_pdf=True | ocr=False | tables=True | converter_cached=False


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

        2 tables, 13 figures, 141 text items in 12.03s


Model files already exist. Using cached files. To redownload, please delete the directory manually: `/teamspace/studios/this_studio/.paddlex/official_models/PP-Chart2Table_safetensors`.
[transformers] The tokenizer you are loading from '/teamspace/studios/this_studio/.paddlex/official_models/PP-Chart2Table_safetensors' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

...

,field,value,rolls_up_to
0,indicators.indicator_type,microbial,
1,indicators.indicator_type,chemical,
2,ingredients.functional_class,carbohydrate,Macronutrient
3,ingredients.functional_class,protein,Macronutrient
4,ingredients.functional_class,fiber,Macronutrient
5,ingredients.functional_class,mineral,Micronutrient
6,ingredients.functional_class,phenol,Bioactive
7,ingredients.functional_class,essential oil,Bioactive
8,ingredients.functional_class,organic acid,Bioactive
9,ingredients.source,animal,



discarded:


,kind,value,occurrences,reason
0,indicator,Cooking loss,40,sensory or gravimetric — not a shelf-life indi...
1,indicator,Sensory score,40,sensory or gravimetric — not a shelf-life indi...
2,indicator,Weight loss,85,sensory or gravimetric — not a shelf-life indi...
3,indicator,hardness,6,sensory or gravimetric — not a shelf-life indi...
4,ingredient,Moringa oleifera Leaf Extract-Treated Ground B...,1,not a known substance — add it to vocabulary.j...
5,ingredient,Moringa oleifera Leaf Extract-Treated Ground B...,1,not a known substance — add it to vocabulary.j...
6,ingredient,Moringa oleifera Leaf Extract-Treated Ground B...,1,not a known substance — add it to vocabulary.j...



review:


,kind,value,reason
0,experiments,Ground beef/Minced beef was packed in parchmen...,dose_belongs_in_concentration: 'Minced beef wa...
1,indicators,Ground beef/Minced beef was packed in parchmen...,indicator_needs_a_unit: no unit in the label a...
2,indicators,Ground beef/Minced beef was packed in parchmen...,indicator_needs_a_unit: no unit in the label a...
3,indicators,Ground beef/Minced beef was packed in parchmen...,indicator_needs_a_unit: no unit in the label a...
4,indicators,Chicken fillet/no protocol/control/unspecified,indicator_needs_a_name: no caption or label na...
5,indicators,Chicken fillet/no protocol/control/unresolved ...,indicator_needs_a_unit: no unit in the label a...
6,indicators,Chicken fillet/no protocol/Thyme essential oil...,indicator_needs_a_name: no caption or label na...
7,indicators,Chicken fillet/no protocol/Thyme essential oil...,indicator_needs_a_unit: no unit in the label a...
8,indicators,Chicken fillet/no protocol/Sage essential oil/...,indicator_needs_a_name: no caption or label na...
9,indicators,Chicken fillet/no protocol/Sage essential oil/...,indicator_needs_a_unit: no unit in the label a...


## Inspecting the Result

Read Gold back out of SQLite, one DataFrame per table. `read_table` is the only reader; the cells below just call it.

In [11]:
import sqlite3

# A separate read-only connection, so inspecting never interferes with a write in
# progress on ENGINE. as_uri() rather than an f-string: a bare "file:C:\..." is not a
# valid URI, so the mode flag would be silently dropped off-Colab.
INSPECT = sqlite3.connect(f"{LOCAL_DB_PATH.as_uri()}?mode=ro", uri=True)

GOLD_TABLES = [
    "papers",
    "sections",
    "tables",
    "figures",
    "experiments",
    "ingredients",
    "experiment_ingredients",
    "indicators",
    "measurements",
    "evidence",
]


def read_table(name: str, limit: int | None = None) -> pd.DataFrame:
    """One table as a DataFrame. The name is checked against the schema rather than
    interpolated blind: SQLite cannot parameterise an identifier, so an allowlist is what
    keeps this from being a string-built query over arbitrary input."""
    if name not in GOLD_TABLES:
        raise ValueError(f"{name!r} is not a Gold table; expected one of {GOLD_TABLES}")
    query = f"SELECT * FROM {name}" + (f" LIMIT {int(limit)}" if limit else "")
    frame = pd.read_sql_query(query, INSPECT)
    print(f"{name}: {len(frame)} rows x {frame.shape[1]} columns")
    return frame

In [12]:
papers = read_table("papers")
papers

papers: 3 rows x 11 columns


,id,doi,title,abstract,published_year,source_path,file_hash,bronze_path,silver_path,markdown_path,created_at
0,1,None,11,None,None,11.pdf,1ef42b74924e18a8c165020bdc174bc36eef4df531e32c...,/content/medallion_extraction/artifacts/bronze/11,/content/medallion_extraction/artifacts/silver/11,/content/medallion_extraction/artifacts/bronze...,2026-07-29 15:49:15.041834
1,2,None,12,None,None,12.pdf,ab392d00aa886dcd514e0e0322a06a042e164dd0442b59...,/content/medallion_extraction/artifacts/bronze/12,/content/medallion_extraction/artifacts/silver/12,/content/medallion_extraction/artifacts/bronze...,2026-07-29 17:11:50.513255
2,3,None,13,None,None,13.pdf,34ef90967c8ed0720ad94cfbee6f87db8f4764b4bcfe58...,/content/medallion_extraction/artifacts/bronze/13,/content/medallion_extraction/artifacts/silver/13,/content/medallion_extraction/artifacts/bronze...,2026-07-29 17:15:23.306149


In [13]:
sections = read_table("sections")
sections

sections: 104 rows x 7 columns


,id,paper_id,section_title,content_markdown,embedding,section_order,docling_item_ref
0,1,1,Document,<!-- image -->\n\nhttp://pubs.acs.org/journal/...,None,1,#/sections/0
1,2,1,Origanum majorana L. Essential Oil-Coated Pape...,"Sulhattin Yasar, Nizam Mustafa Nizaml ı og ̆ l...",None,2,#/sections/1
2,3,1,ACCESS,[Metrics &amp; More](https://pubs.acs.org/doi/...,None,3,#/sections/2
3,4,1,■ INTRODUCTION,During refrigeration of processed meat product...,None,4,#/sections/3
4,5,1,■ MATERIALS AND METHODS,Materials. OmEO was purchased from a commercia...,None,5,#/sections/4
...,...,...,...,...,...,...,...
99,100,3,3.2.2. Antimicrobial Effect of MLE against S. ...,There was no significant difference ( p &gt; 0...,None,15,#/sections/14
100,101,3,3.2.3. Antimicrobial Effect of MLE against S. ...,The current study revealed no significant diff...,None,16,#/sections/15
101,102,3,3.3. Sensory Evaluation,Moringa oleifera is a nutraceutical element ri...,None,17,#/sections/16
102,103,3,References,"1. Behbahani, B.A.; Noshad, M.; Jooyandeh, H. ...",None,18,#/sections/17


In [14]:
tables = read_table("tables")
tables

tables: 7 rows x 7 columns


,id,paper_id,caption,csv_filepath,structured_json,docling_item_ref,created_at
0,1,1,"Table 2. Changes in Hardness, pH, HCl Titrate,...",/content/medallion_extraction/artifacts/silver...,"{""gate"": {""fits"": false, ""reason"": ""no column ...",#/tables/1,2026-07-29 15:49:15.046332
1,2,1,"Table 2. Changes in Hardness, pH, HCl Titrate,...",/content/medallion_extraction/artifacts/silver...,"{""gate"": {""fits"": false, ""reason"": ""no column ...",#/tables/1,2026-07-29 16:34:45.595955
2,3,1,"Table 2. Changes in Hardness, pH, HCl Titrate,...",/content/medallion_extraction/artifacts/silver...,"{""gate"": {""fits"": false, ""reason"": ""no column ...",#/tables/1,2026-07-29 16:50:27.261946
3,4,1,"Table 2. Changes in Hardness, pH, HCl Titrate,...",/content/medallion_extraction/artifacts/silver...,"{""gate"": {""fits"": true, ""reason"": ""keyed: the ...",#/tables/1,2026-07-29 16:58:07.661677
4,5,2,Table 1 Chemical composition of TEO and SEO me...,/content/medallion_extraction/artifacts/silver...,"{""gate"": {""fits"": false, ""reason"": ""no column ...",#/tables/0,2026-07-29 17:11:50.911097
5,6,3,Table 1. Effect of Moringa oleifera leaves ext...,/content/medallion_extraction/artifacts/silver...,"{""gate"": {""fits"": true, ""reason"": ""long: one a...",#/tables/0,2026-07-29 17:15:23.310279
6,7,3,Table 2. * Mean values of the sensory characte...,/content/medallion_extraction/artifacts/silver...,"{""gate"": {""fits"": true, ""reason"": ""keyed: the ...",#/tables/1,2026-07-29 17:15:23.310284


In [15]:
figures = read_table("figures")
figures

figures: 14 rows x 7 columns


,id,paper_id,caption,image_filepath,semantic_tags_json,docling_item_ref,created_at
0,1,1,"Figure 3. DPPH scavenging activity (%), degree...",/content/medallion_extraction/artifacts/bronze...,[],#/pictures/11,2026-07-29 16:58:07.528143
1,2,2,None,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/3,2026-07-29 17:11:50.814418
2,3,2,Fig. 2. Changes in TVB-N values of chicken fil...,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/4,2026-07-29 17:11:50.814425
3,4,2,Fig. 3. Changes in PV values of chicken ...,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/5,2026-07-29 17:11:50.814429
4,5,2,None,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/6,2026-07-29 17:11:50.814432
5,6,2,None,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/7,2026-07-29 17:11:50.814435
6,7,2,None,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/8,2026-07-29 17:11:50.814438
7,8,2,Fig. 6. Changes in percentage of cooking loss ...,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/9,2026-07-29 17:11:50.814441
8,9,2,Fig. 7. Sensory evaluation of chicken fillets....,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/10,2026-07-29 17:11:50.814444
9,10,2,None,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/11,2026-07-29 17:11:50.814447


In [16]:
experiments = read_table("experiments")
experiments

experiments: 16 rows x 4 columns


,experiment_id,meat_matrix,treatment,weight_g
0,1,Ground beef,Minced beef was packed in parchment paper coat...,NaN
1,2,Ground beef,Minced beef was packed in parchment paper coat...,NaN
2,3,Ground beef,Minced beef was packed in parchment paper coat...,NaN
3,4,Chicken fillet,None,NaN
4,5,Chicken fillet,None,NaN
5,6,Chicken fillet,None,NaN
6,7,Chicken fillet,None,NaN
7,8,Chicken fillet,None,NaN
8,9,Chicken fillet,None,NaN
9,10,Chicken fillet,None,NaN


In [17]:
ingredients = read_table("ingredients")
ingredients

ingredients: 31 rows x 4 columns


,ingredient_id,ingredient_name,functional_class,source
0,1,Thyme essential oil,essential oil,plant
1,2,Sage essential oil,essential oil,plant
2,3,α-Thujene,essential oil,plant
3,4,α-Pinene,essential oil,plant
4,5,Camphene,essential oil,plant
5,6,β-Pinene,essential oil,plant
6,7,3-Octanone,essential oil,plant
7,8,Myrcene,essential oil,plant
8,9,α-Terpinene,essential oil,plant
9,10,p-Cymene,essential oil,plant


In [18]:
experiment_ingredients = read_table("experiment_ingredients")
experiment_ingredients

experiment_ingredients: 39 rows x 4 columns


,experiment_id,ingredient_id,concentration,concentration_unit
0,5,1,2.00,%
1,6,1,1.00,%
2,7,2,2.00,%
3,8,2,1.00,%
4,9,1,2.00,%
5,9,2,2.00,%
6,10,1,1.00,%
7,10,2,1.00,%
8,11,3,0.20,%
9,11,4,2.21,%


In [19]:
indicators = read_table("indicators")
indicators

indicators: 21 rows x 5 columns


,indicator_id,indicator_name,indicator_type,indicator_unit,indicator_threshold
0,1,pH,chemical,pH,NaN
1,2,HCl titrate value,chemical,mL/g,NaN
2,3,DM,chemical,%,NaN
3,4,Water activity,chemical,aw,NaN
4,5,Total viable count,microbial,log CFU/g,7.0
5,6,"DPPH scavenging activity, %",chemical,unspecified,NaN
6,7,"TBA, gram MAD/kg dry matter",chemical,unspecified,NaN
7,8,"Peroxide value, mEq/kg sample",chemical,unspecified,NaN
8,9,unresolved indicator,chemical,unspecified,NaN
9,10,Total volatile basic nitrogen,chemical,mg N/100 g,25.0


In [20]:
measurements = read_table("measurements")
measurements

measurements: 366 rows x 4 columns


,experiment_id,day,indicator_id,indicator_value
0,1,0,1,5.64
1,1,0,2,0.10
2,1,0,3,34.39
3,1,0,4,0.96
4,1,0,5,2.47
...,...,...,...,...
361,16,6,18,6.61
362,16,6,19,7.18
363,16,6,21,7.09
364,16,6,20,7.24


In [21]:
evidence = read_table("evidence")
evidence

evidence: 84 rows x 14 columns


,id,paper_id,entity_type,entity_key,field_name,page_number,source_type,source_label,exact_text,method,rationale,confidence,value_is_approximate,docling_item_ref
0,1,1,experiment,"{""experiment_id"": 1}",indicator_value,6.0,table,None,None,stated,None,0.5,0,#/tables/1
1,2,1,experiment,"{""experiment_id"": 1}",treatment,NaN,prose,None,"The coating process involved two steps. First,...",stated,The text describes application of the starch s...,0.5,0,#/sections/4
2,3,1,experiment,"{""experiment_id"": 1}",weight_g,NaN,prose,None,,inferred,No mass of a single minced beef unit is report...,0.3,0,#/sections/4
3,4,1,experiment,"{""experiment_id"": 2}",indicator_value,6.0,table,None,None,stated,None,0.5,0,#/tables/1
4,5,1,experiment,"{""experiment_id"": 2}",treatment,NaN,prose,None,"The temperature was then lowered to 70 °C, dur...",stated,The text specifies that OmEO was incorporated ...,0.5,0,#/sections/4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,80,3,experiment,"{""experiment_id"": 16}",indicator_value,10.0,table,None,None,stated,None,0.5,0,#/tables/1
80,81,3,experiment,"{""experiment_id"": 16}",indicator_value,10.0,table,None,None,stated,None,0.5,0,#/tables/1
81,82,3,experiment,"{""experiment_id"": 16}",indicator_value,10.0,table,None,None,stated,None,0.5,0,#/tables/1
82,83,3,experiment,"{""experiment_id"": 16}",indicator_value,10.0,table,None,None,stated,None,0.5,0,#/tables/1


In [22]:
INSPECT.close()
print("Inspection connection closed.")

Inspection connection closed.
